# 세 가지 RAG 파이프라인 비교 노트북 (전체 500건)

이 노트북은 `windows_project_driver.ipynb`와 같은 프로젝트(`sct-project-graphrag`)를 사용하되,
**5건짜리 mini corpus가 아니라 전체 500건 데이터**에 대해 세 파이프라인의 답변을 한 번에 비교합니다.

| | 파이프라인 | 실행 스크립트 | 입력 데이터 |
|---|---|---|---|
| 1 | **Chunk BM25** (baseline) | `scripts/09_bm25_raw_rag_answer.py` | `data/processed/pdf_pages.jsonl` |
| 2 | **Issue-based** (structural) | `scripts/10_bm25_issue_rag_answer.py --raw-context same-case` | `data/processed/case_issues.jsonl` (+ pages) |
| 3 | **Graph RAG** (relational) | `scripts/14_graph_rag_answer.py` | `data/indexes/issue_graph_with_similarity.json` |

세 파이프라인 모두 전처리 산출물(500건)이 이미 repo에 들어 있으므로, 아래 **3~5절을 순서대로 한 번만** 실행한 뒤,
마지막 6절에서 `compare("질문")` 셀을 복사해 **질문 문자열만 바꿔가며** 여러 개를 테스트하면 됩니다.

## 1. 프로젝트 폴더 경로 설정

`PROJECT_DIR`만 본인 컴퓨터의 프로젝트 폴더 경로로 고치세요. Windows 경로 앞에는 `r`을 붙입니다.

In [1]:
from pathlib import Path
import os
import sys

# 예: r"C:\\sct-project-graphrag"  /  r"D:\\snu_python\\graphrag"
PROJECT_DIR = Path(r"D:\snu_python\graphrag")

# 전체 파이프라인이므로 scripts/ 와 data/ 가 있어야 합니다.
REQUIRED_MARKERS = ["pyproject.toml", "scripts", "data"]

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"프로젝트 폴더를 찾지 못했습니다: {PROJECT_DIR}")

missing = [m for m in REQUIRED_MARKERS if not (PROJECT_DIR / m).exists()]
if missing:
    raise RuntimeError(
        f"프로젝트 폴더로 보이지 않습니다: {PROJECT_DIR}\n"
        f"없는 항목: {missing}\n"
        "zip이 한 겹 더 들어가서 풀렸는지 확인하고 PROJECT_DIR을 안쪽 폴더로 고치세요."
    )

os.chdir(PROJECT_DIR)
print("프로젝트 폴더:", PROJECT_DIR)
print("현재 작업 폴더:", Path.cwd())
print("노트북 Python:", sys.executable)

프로젝트 폴더: D:\snu_python\graphrag
현재 작업 폴더: d:\snu_python\graphrag
노트북 Python: c:\Users\Dell\miniforge3\envs\openai\python.exe


## 2. 패키지 설치 (이미 했다면 건너뛰어도 됩니다)

`windows_project_driver.ipynb`에서 한 번 설치했다면 다시 실행하지 않아도 됩니다.

In [ ]:
# %pip install --upgrade pip
# %pip install -e ".[graph]"

## 3. API 환경변수 설정

학내 Wi-Fi에 연결된 상태에서 실행하세요. `.env` 파일은 `PROJECT_DIR/.env`에 있어야 하며,
세 스크립트 모두 이 `.env`를 읽습니다.

In [2]:
def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        raise FileNotFoundError(f".env 파일을 찾지 못했습니다: {env_path}")
    with env_path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

ENV_FILE = PROJECT_DIR / ".env"
load_env_file(ENV_FILE)

required = ["OPENAI_API_KEY", "BASE_URL", "OPENAI_MODEL", "OPENAI_REASONING_EFFORT", "OPENAI_EMBEDDING_MODEL"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f".env에 없는 값: {missing}. {ENV_FILE} 파일을 확인하세요.")

print(f"{ENV_FILE}에서 API 환경변수를 불러왔습니다.")
print("BASE_URL     :", os.environ["BASE_URL"])
print("OPENAI_MODEL :", os.environ["OPENAI_MODEL"])

D:\snu_python\graphrag\.env에서 API 환경변수를 불러왔습니다.
BASE_URL     : ldi.snu.ac.kr:30000
OPENAI_MODEL : gpt-5.4-mini # gpt-5.3-codex-spark, gpt-5.4-mini'# gpt-5.4-mini


## 4. 전체 500건 산출물 확인

세 파이프라인이 사용할 전처리 결과가 `data/` 아래에 있는지 점검합니다.
repo에는 500건 처리 결과가 이미 포함되어 있으므로, 보통은 아래 셀이 모두 `OK`로 나옵니다.

In [3]:
DATA = PROJECT_DIR / "data"
SCRIPTS_DIR = PROJECT_DIR / "scripts"

PDF_PAGES   = DATA / "processed" / "pdf_pages.jsonl"                 # 파이프라인 1, 2
CASE_ISSUES = DATA / "processed" / "case_issues.jsonl"               # 파이프라인 2
SIM_GRAPH   = DATA / "indexes"   / "issue_graph_with_similarity.json"  # 파이프라인 3

def _count_lines(p):
    try:
        with p.open(encoding="utf-8") as f:
            return sum(1 for _ in f)
    except Exception:
        return "?"

required_files = {
    "pdf_pages.jsonl   (P1, P2)": PDF_PAGES,
    "case_issues.jsonl (P2)":     CASE_ISSUES,
    "issue_graph_with_similarity.json (P3)": SIM_GRAPH,
}

all_ok = True
for label, path in required_files.items():
    if path.exists():
        extra = f" lines={_count_lines(path)}" if path.suffix == ".jsonl" else ""
        print(f"OK    {label}{extra}")
    else:
        all_ok = False
        print(f"MISS  {label}  ->  {path}")

if all_ok:
    print("\n전처리 산출물이 모두 준비되었습니다. 5절로 넘어가세요.")
else:
    print("\n누락된 파일이 있습니다. 4-b절(선택)에서 해당 단계를 다시 생성하세요.")

OK    pdf_pages.jsonl   (P1, P2) lines=3585
OK    case_issues.jsonl (P2) lines=500
OK    issue_graph_with_similarity.json (P3)

전처리 산출물이 모두 준비되었습니다. 5절로 넘어가세요.


In [ ]:
from sct_graphrag.io import load_jsonl, group_pages_by_document, iter_issue_records
import json
import subprocess

MANIFEST = DATA / "processed" / "sample_cases.jsonl"
PDF_DIR  = DATA / "raw" / "pdfs"

# 1) 매니페스트 / 내려받은 PDF
manifest_rows = load_jsonl(MANIFEST) if MANIFEST.exists() else []
pdf_files = sorted(PDF_DIR.glob("*.pdf")) if PDF_DIR.exists() else []

# 2) 페이지 텍스트 → 사건 수
pages = load_jsonl(PDF_PAGES)
page_docs = group_pages_by_document(pages)

# 3) 쟁점 레코드 → 사건 수
issues = list(iter_issue_records(load_jsonl(CASE_ISSUES)))
issue_docs = {r["document_id"] for r in issues}

# 4) 그래프 → Issue 노드와 연결된 사건 수
raw = json.loads(SIM_GRAPH.read_text(encoding="utf-8"))
nodes = raw.get("nodes", raw) if isinstance(raw, dict) else raw
nodes = list(nodes.values()) if isinstance(nodes, dict) else nodes
from collections import Counter
type_counts = Counter(n.get("type") for n in nodes)
issue_nodes = [n for n in nodes if n.get("type") == "Issue"]
graph_docs = {n.get("document_id") for n in issue_nodes if n.get("document_id")}

print(f"① manifest 사건 수        : {len(manifest_rows)}")
print(f"② 내려받은 PDF 파일 수    : {len(pdf_files)}")
print(f"③ pdf_pages 사건 수       : {len(page_docs)}   (총 페이지 {len(pages)})")
print(f"④ case_issues 사건 수     : {len(issue_docs)}   (총 쟁점 {len(issues)})")
print(f"⑤ graph Issue-연결 사건 수: {len(graph_docs)}   (Issue 노드 {len(issue_nodes)})")
print("  graph 노드 타입 분포     :", dict(type_counts))

# 일치 여부 판정
counts = [len(page_docs), len(issue_docs), len(graph_docs)]
print("\n→ 사건 수 일치 여부:", "OK (정합)" if len(set(counts)) == 1 else f"불일치 {counts} — 가장 작은 단계가 실패한 지점")

# 내용이 깨지지 않았는지 한 건 눈으로 확인
print("\n[샘플 쟁점 레코드]")
s = issues[0]
print("  file     :", s.get("document_id"))
print("  쟁점     :", (s.get("issue") or "")[:80])
print("  결론     :", (s.get("conclusion") or "")[:80])

① manifest 사건 수        : 500
② 내려받은 PDF 파일 수    : 0
③ pdf_pages 사건 수       : 500   (총 페이지 3585)
④ case_issues 사건 수     : 500   (총 쟁점 701)
⑤ graph Issue-연결 사건 수: 500   (Issue 노드 701)
  graph 노드 타입 분포     : {'Case': 500, 'Issue': 701, 'Outcome': 17, 'LegalConcept': 2148, 'FactPattern': 3598, 'EvidenceType': 2073}

→ 사건 수 일치 여부: OK (정합)

[샘플 쟁점 레코드]
  file     : 국심-1999-경-0644.pdf
  쟁점     : 쟁점세금계산서가 실지거래내용에 따라 실지거래상대방으로부터 교부받은 세금계산서인지, 아니면 사실과 다른 세금계산서인지 여부
  결론     : 기각. 쟁점세금계산서는 사실과 다른 세금계산서로 보아 관련 매입세액을 불공제한 처분이 정당하다.


## 4-b. (선택) 전체 파이프라인 재생성

위 4절에서 누락된 파일이 있거나, 500건 전처리를 처음부터 직접 만들고 싶을 때만 사용합니다.

> ⚠️ `04`(issue 추출)와 `11`(graph feature 추출)은 500건 전체에 대해 LLM을 호출하므로 시간과 API 사용량이 큽니다.
> 수업 중에는 repo에 포함된 500건 결과를 그대로 쓰는 것을 권장합니다.

아래 셀은 기본적으로 **실행되지 않도록** `RUN_FULL_BUILD = False`로 막아두었습니다.
필요한 단계만 선택적으로 켜서 돌리세요.

In [ ]:
# import subprocess

# def run_full(script_name, *args):
#     """scripts/ 아래 전체(500건) 스크립트를 실행하고 출력을 실시간으로 보여줍니다."""
#     cmd = [sys.executable, str(SCRIPTS_DIR / script_name), *map(str, args)]
#     print("$ " + " ".join(cmd))
#     proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
#                             stderr=subprocess.STDOUT, text=True, bufsize=1)
#     for line in proc.stdout:
#         print(line, end="")
#     proc.wait()
#     if proc.returncode != 0:
#         raise subprocess.CalledProcessError(proc.returncode, cmd)

# RUN_FULL_BUILD = False  # 전체 재생성이 필요할 때만 True 로 바꾸세요.

# if RUN_FULL_BUILD:
#     # 01~03: 메타데이터 -> PDF 500개 -> 페이지 텍스트
#     run_full("01_download_case_metadata.py", "--overwrite")
#     run_full("02_download_sample_pdfs.py")
#     run_full("03_extract_pdf_text.py", "--overwrite")
#     # 04~05: issue 추출(LLM) -> issue embedding index  (API 사용, 비용 큼)
#     run_full("04_extract_case_issues.py", "--overwrite", "--concurrency", "3")
#     run_full("05_build_issue_embedding_index.py")
#     # 11~12: graph feature 추출(LLM) -> typed graph + similarity graph  (API 사용, 비용 큼)
#     run_full("11_extract_graph_features.py", "--overwrite", "--concurrency", "3")
#     run_full("12_build_issue_graph.py")
#     run_full("12_build_issue_graph.py", "--add-similarity",
#              "--output", str(SIM_GRAPH))
#     print("\n전체 파이프라인 재생성 완료")
# else:
#     print("RUN_FULL_BUILD=False 이므로 재생성을 건너뜁니다. (repo의 500건 결과 사용)")

## 5. 세 파이프라인 비교 harness

`compare(query)`는 같은 질문을 세 파이프라인에 각각 넣고, 각 스크립트의 `## Answer` 부분을 뽑아 나란히 보여줍니다.
내부적으로는 전체 500건 데이터 경로를 명시적으로 넘겨 실행합니다.

옵션:
- `top_k_raw` : 파이프라인 1에서 가져올 raw chunk 수 (기본 10)
- `top_k_issue` : 파이프라인 2에서 가져올 issue 수 (기본 5)
- `graph_mode` : 파이프라인 3 분석 모드 — `"auto"`(질문에 따라 자동) / `"outcome-comparison"` / `"overview"`
- `show_logs` : `True`면 각 파이프라인이 무엇을 검색했는지(retrieval log)도 함께 표시
- `which` : 일부 파이프라인만 돌리고 싶을 때 예) `which=(1, 3)`

In [9]:
import subprocess

In [4]:
import time

PIPELINE_TITLES = {
    1: "파이프라인 1 · Chunk BM25 (baseline)",
    2: "파이프라인 2 · Issue-based (structural)",
    3: "파이프라인 3 · Graph RAG (relational)",
}

def _run(script_name, *args, timeout=900):
    cmd = [sys.executable, str(SCRIPTS_DIR / script_name), *map(str, args)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True, timeout=timeout)
    elapsed = time.time() - t0
    out = proc.stdout or ""
    if proc.returncode != 0:
        out += "\n\n[stderr]\n" + (proc.stderr or "")
    return proc.returncode, out, elapsed

def _split_answer(stdout):
    """(retrieval_log, answer_text) 로 분리.
    '## Answer' 가 단독 헤더 줄로 처음 나오는 지점에서 자른다.
    답변 본문에 들어 있는 ###/## 헤더에 오작동하지 않도록 줄 단위로 매칭."""
    lines = stdout.splitlines()
    for i, line in enumerate(lines):
        if line.strip() == "## Answer":
            log = "\n".join(lines[:i]).strip()
            answer = "\n".join(lines[i + 1:]).strip()
            return log, answer
    return stdout.strip(), ""

def _run_pipeline(n, query, top_k_raw, top_k_issue, graph_mode):
    if n == 1:
        return _run("09_bm25_raw_rag_answer.py", query,
                    "--input", PDF_PAGES, "--top-k", str(top_k_raw),
                    "--env-file", ENV_FILE)
    if n == 2:
        return _run("10_bm25_issue_rag_answer.py", query,
                    "--issues", CASE_ISSUES, "--pages", PDF_PAGES,
                    "--top-k", str(top_k_issue), "--raw-context", "same-case",
                    "--env-file", ENV_FILE)
    if n == 3:
        return _run("14_graph_rag_answer.py", query,
                    "--graph", SIM_GRAPH, "--analysis-mode", graph_mode,
                    "--env-file", ENV_FILE)
    raise ValueError(n)

def _render(query, results, show_logs):
    try:
        from IPython.display import Markdown, display
        use_md = True
    except Exception:
        use_md = False

    if use_md:
        md = [f"### 질문\n\n> {query}\n"]
        for n, log, answer, elapsed, code in results:
            status = "✅" if code == 0 else "⚠️"
            md.append(f"### {status} {PIPELINE_TITLES[n]}  · _{elapsed:.1f}s_\n\n{answer or '(답변 없음)'}\n")
            if show_logs and log:
                snippet = log[:6000]
                md.append("<details><summary>retrieval log 보기</summary>\n\n```\n"
                          + snippet + "\n```\n</details>\n")
            md.append("\n---\n")
        display(Markdown("\n".join(md)))
    else:
        print("=" * 90)
        print("질문:", query)
        for n, log, answer, elapsed, code in results:
            print("\n" + "=" * 90)
            print(f"{PIPELINE_TITLES[n]}  ({elapsed:.1f}s, exit={code})")
            print("-" * 90)
            if show_logs and log:
                print(log[:4000]); print("-" * 90)
            print(answer or "(답변 없음)")

def compare(query, *, top_k_raw=10, top_k_issue=5, graph_mode="auto",
            show_logs=False, which=(1, 2, 3)):
    results = []
    for n in which:
        print(f"\u25b6 {PIPELINE_TITLES[n]} 실행 중...", flush=True)
        try:
            code, out, elapsed = _run_pipeline(n, query, top_k_raw, top_k_issue, graph_mode)
            log, answer = _split_answer(out)
            results.append((n, log, answer, elapsed, code))
        except Exception as e:
            results.append((n, "", f"[실행 실패] {e!r}", 0.0, -1))
    _render(query, results, show_logs)
    return results

print("compare() 준비 완료")

compare() 준비 완료


## 5-b. Retrieval 성능 지표 (label-free)

`compare(...)`가 돌려준 결과만 가지고, 정답 라벨 없이 바로 계산할 수 있는 지표 3가지를 봅니다.

- **score 분포 / 확신도** : top-1 score가 top-k 평균보다 얼마나 높은지 (margin이 클수록 검색이 한 사건에 "확신"을 가졌다는 뜻)
- **사건 겹침도 (Jaccard)** : 세 파이프라인이 찾아온 사건(파일) 집합이 서로 얼마나 겹치는지. 겹침이 낮으면 같은 질문에도 완전히 다른 사건을 근거로 답했다는 뜻
- **Latency** : 파이프라인별 실행 시간 (`compare()`가 이미 측정한 `elapsed`를 표로 정리)

`retrieval log`(각 스크립트의 `## Retrieved ...` / `## Representative Cases` 출력)를 정규식으로 파싱해서 `(score, file_id)` 쌍을 뽑습니다.
별도 정답 데이터가 필요 없어서 모든 쿼리에 바로 적용할 수 있습니다.

In [5]:
import re
from statistics import mean

# 파이프라인별 retrieval log 라인 포맷
#   P1 (09_bm25_raw_rag_answer.py) : "1. score=0.123 file=foo.pdf chunk=2"
#   P2 (10_bm25_issue_rag_answer.py): "1. score=0.123 file=foo.pdf issue=0"
#   P3 (14_graph_rag_answer.py)     : "  [A1] file=foo.pdf issue=0 outcome=인용 score=12.34"
_P1_P2_LINE = re.compile(r"score=([\d.]+)\s+file=(\S+?)(?:\.pdf)?\s+(?:chunk|issue)=")
_P3_LINE = re.compile(r"\[[A-Z]\d+\]\s+file=(\S+?)(?:\.pdf)?\s+issue=\S+\s+outcome=\S+\s+score=([\d.]+)")


def _parse_retrieval_log(n, log):
    # log 텍스트에서 (score, case_id) 쌍 리스트를 뽑는다. 못 찾으면 빈 리스트.
    pairs = []
    if not log:
        return pairs
    if n in (1, 2):
        for m in _P1_P2_LINE.finditer(log):
            score, case_id = float(m.group(1)), m.group(2)
            pairs.append((score, case_id))
    elif n == 3:
        for m in _P3_LINE.finditer(log):
            case_id, score = m.group(1), float(m.group(2))
            pairs.append((score, case_id))
    return pairs


def _score_stats(pairs, top_k=5):
    # top-1 score, top-k 평균, margin(top1 - top-k 평균)을 계산
    if not pairs:
        return {"top1": None, "topk_mean": None, "margin": None, "n": 0}
    scores = [s for s, _ in pairs[:top_k]]
    top1 = scores[0]
    topk_mean = mean(scores)
    margin = top1 - topk_mean
    return {"top1": round(top1, 3), "topk_mean": round(topk_mean, 3), "margin": round(margin, 3), "n": len(pairs)}


def _jaccard(set_a, set_b):
    if not set_a and not set_b:
        return None
    union = set_a | set_b
    if not union:
        return None
    return len(set_a & set_b) / len(union)


def compute_metrics(results, top_k=5, show_log_excerpt=False):
    # results: compare()가 반환한 [(n, log, answer, elapsed, code), ...]
    parsed = {}
    for n, log, answer, elapsed, code in results:
        pairs = _parse_retrieval_log(n, log)
        case_ids = {case_id for _, case_id in pairs}
        parsed[n] = {
            "pairs": pairs,
            "case_ids": case_ids,
            "elapsed": elapsed,
            "score_stats": _score_stats(pairs, top_k=top_k),
        }
        if show_log_excerpt:
            print(f"-- P{n} retrieval log 일부 (파싱된 {len(pairs)}건) --")
            print((log or "")[:400])
            print()

    try:
        from IPython.display import Markdown, display
        use_md = True
    except Exception:
        use_md = False

    lines = []
    lines.append("| 파이프라인 | top-1 score | top-{0} 평균 | margin | 검색 건수 | latency(s) |".format(top_k))
    lines.append("|---|---|---|---|---|---|")
    for n in sorted(parsed):
        s = parsed[n]["score_stats"]
        lines.append(
            f"| P{n} | {s['top1']} | {s['topk_mean']} | {s['margin']} | {s['n']} | {parsed[n]['elapsed']:.1f} |"
        )
    score_table = "\n".join(lines)

    overlap_lines = ["| 비교 | 겹친 사건 수 | Jaccard |", "|---|---|---|"]
    ns = sorted(parsed)
    for i in range(len(ns)):
        for j in range(i + 1, len(ns)):
            a, b = ns[i], ns[j]
            set_a, set_b = parsed[a]["case_ids"], parsed[b]["case_ids"]
            jac = _jaccard(set_a, set_b)
            inter = len(set_a & set_b)
            jac_str = f"{jac:.2f}" if jac is not None else "n/a"
            overlap_lines.append(f"| P{a} ∩ P{b} | {inter} | {jac_str} |")
    overlap_table = "\n".join(overlap_lines)

    if use_md:
        display(Markdown("**Score 분포 / 확신도**\n\n" + score_table))
        display(Markdown("**사건 겹침도 (Jaccard)**\n\n" + overlap_table))
    else:
        print("Score 분포 / 확신도")
        print(score_table)
        print()
        print("사건 겹침도 (Jaccard)")
        print(overlap_table)

    return parsed


print("compute_metrics() 준비 완료")

compute_metrics() 준비 완료


## 6. 쿼리 테스트

아래 셀들은 `queries.md`의 세 가지 질문 유형(키워드형 / 상담형 / 패턴형)을 예시로 넣어 둔 것입니다.

**셀을 복사한 뒤 `compare("...")` 안의 질문 문자열만 바꿔서** 원하는 만큼 테스트하세요.
각 파이프라인이 어떤 질문에서 강점을 보이는지 비교하면 됩니다.

### 6-1. 키워드형 질문 (파이프라인 1이 비교적 잘 잡는 유형)

질문 표현이 결정문 원문에 그대로 등장하는 경우입니다.

In [ ]:
results = compare("사실과 다른 세금계산서가 문제된 사건들에서 매입세액 공제가 부인된 이유는 뭐야?")

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Issue-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...


### 질문

> 사실과 다른 세금계산서가 문제된 사건들에서 매입세액 공제가 부인된 이유는 뭐야?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _36.3s_

사실과 다른 세금계산서가 문제된 사건들에서 매입세액 공제가 부인된 핵심 이유는, **세금계산서가 실제 거래 내용과 일치하지 않거나, 법정 발급시기를 지나 발급·수취되었기 때문**입니다. 구체적으로는 다음과 같습니다.

1. **필요적 기재사항이 사실과 다른 경우**
   - 거래당사자, 목적물, 가액, 시기 등이 실제 거래와 일치하지 않으면 “사실과 다른 세금계산서”로 보아 매입세액 공제가 부인됩니다. [2]
   - 예를 들어, 실제 공급시기와 세금계산서 작성일자가 맞지 않거나, 실제 공급자와 세금계산서상 공급자가 다른 경우가 여기에 해당합니다. [2], [6]

2. **공급시기가 속하는 과세기간·신고기간을 지나 세금계산서를 받은 경우**
   - 공급시기 이후 너무 늦게 세금계산서를 받으면, 부가가치세법상 “세금계산서를 발급받지 아니한 경우”로 보아 매입세액 공제가 거부될 수 있습니다. [1]
   - 특히 공급시기 또는 발급기한을 넘겨 수취한 경우 매입세액을 불공제한다는 취지의 설명이 나옵니다. [1]

3. **실질거래가 없는 가공거래 또는 명목상 거래로 본 경우**
   - 재화의 실제 인도·양도가 없는 거래, 폭탄업체·자료상 등을 거친 거래는 사실과 다른 세금계산서로 보아 매입세액 공제가 부인됩니다. [3], [5]
   - 이런 경우 세금계산서가 형식상 존재해도, 실제 거래를 뒷받침하지 못하면 공제가 인정되지 않습니다. [3], [5]

4. **공사 시행주체가 아닌 다른 사업자로부터 받은 세금계산서인 경우**
   - 건물 신축 등에서 실제 공사를 시행한 주체가 아닌 다른 사업자로부터 받은 세금계산서는 사실과 다른 세금계산서로 보아 매입세액 공제가 부인됩니다. [6], [8]

정리하면, 이 사건들에서는 **세금계산서의 형식보다 실질 거래와의 일치 여부**, 그리고 **법정 발급시기 준수 여부**가 중요했고, 그 요건을 충족하지 못해 매입세액 공제가 부인되었습니다. [1], [2], [6]

원하시면 제가 이걸 **“판례형 한 문장 답변”**으로 더 짧게 정리해드릴게요.


---

### ✅ 파이프라인 2 · Issue-based (structural)  · _30.2s_

사실과 다른 세금계산서 사건에서 **매입세액 공제가 부인된 핵심 이유는, 세금계산서가 실제 거래를 반영하지 못했기 때문**입니다. 구체적으로는 다음과 같습니다.

1. **실제 공급자와 세금계산서상 공급자가 다른 경우**
   - 실제로 건설용역이나 재화를 공급한 자와, 세금계산서에 적힌 공급자가 일치하지 않으면 「사실과 다른 세금계산서」로 보아 매입세액 공제가 배제되었습니다.[1][4]
   - 예를 들어, 쟁점건물의 실제 시행주체는 다른 사업자로 보이는데도 그와 다른 사업자로부터 세금계산서를 수취한 경우가 이에 해당했습니다.[1]

2. **공급받은 내용과 세금계산서 기재 내용이 다를 때**
   - 미완성 건물을 양수한 경우라면 기성고 부분과 추가공사 부분을 구분해 공급자로부터 적법하게 세금계산서를 받아야 하는데, 이를 구분하지 않고 다른 내용으로 수취하면 사실과 다른 세금계산서로 판단되었습니다.[1]

3. **공급시기와 수취시기가 맞지 않는 경우**
   - 용역의 공급시기보다 늦게 세금계산서를 받았고, 그 시점이 과세기간도 달라서 법정 매입세액 공제 요건을 충족하지 못한 사례가 있었습니다.[2][3]
   - 즉, 건물 사용승인일을 공급시기로 보았는데 세금계산서 수취일이 그 이후 과세기간에 속하면 공제가 안 됩니다.[2]

4. **거래의 실질이 가공거래로 보이는 경우**
   - 자료상으로 확인되거나 실제 거래 없이 세금계산서만 발행·수취한 경우에는 매입세액 공제가 인정되지 않았습니다.[4][5]
   - 유류·고철 등 거래에서도 사업장 실재성, 물량 출처, 자금흐름 등 실질 확인이 부족하면 가공거래로 판단될 수 있었습니다.[4][5]

5. **선의·무과실이 인정되지 않는 경우**
   - 거래상대방의 실재나 정상거래 여부를 충분히 확인하지 않았고, 주의의무를 다했다고 보기 어려우면 선의의 거래당사자로 보지 않아 공제가 부인되었습니다.[4][5]

정리하면, 이런 사건들에서는 **“실제 공급자 불일치” + “거래 실질 불명확” + “공급시기/과세기간 불일치”**가 주된 부인 사유였습니다.[1][2][4][5]

원하시면 제가 이걸 **한 줄 요약형**이나 **판례별 비교표**로도 정리해드릴게요.


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _42.1s_

그래프 컨텍스트상, **사실과 다른 세금계산서 사건에서 매입세액 공제가 부인된 핵심 이유**는 대체로 다음과 같습니다.

1. **실제 공급자와 세금계산서상의 공급자가 다른 경우**
   - 명의상 거래처가 실제 공급자가 아니어서, 세금계산서의 공급자 기재가 사실과 다르다고 본 사례가 반복됩니다.  
   - 예: 실제 공급자가 다른 회사로 조사된 경우 [C1], 명의상 거래상대방이 실제 용역공급자가 아니라고 본 경우 [C4], 공급자가 사실과 다른 세금계산서로 본 경우 [C3].

2. **실물거래 없이 세금계산서만 수수된 가공거래로 인정된 경우**
   - 거래 자체가 허위이거나 실재하지 않아, 세금계산서가 거래의 실질을 반영하지 못했다고 보아 매입세액 공제를 부인했습니다.  
   - 예: 실물거래 없이 세금계산서만 수수된 가공거래 [C2].

3. **거래처가 자료상으로 확인·고발된 경우**
   - 상대방이 자료상으로 확정되거나 고발되면, 세금계산서의 신빙성이 크게 흔들리고 사실과 다른 세금계산서로 판단되는 패턴이 많습니다.  
   - 그래프에서도 `거래처가 자료상으로 확정됨`, `거래처가 자료상으로 고발됨`, `청구외법인이 자료상혐의자로 조사되어 고발됨`이 반복됩니다. [C4]

4. **청구인이 선의의 거래당사자라는 점을 입증하지 못한 경우**
   - 단순히 사업자등록증, 법인인감, 세금계산서, 입금증 등을 갖췄다는 사정만으로는 부족하고,  
     실제 공급자를 확인하려는 **선량한 관리자의 주의의무**를 다했는지가 중요하게 보였습니다.
   - 주의의무를 다하지 못했다고 본 경우 매입세액 공제가 부인되었습니다.  
   - 예: 기본서류 확인만으로는 부족하다고 본 사례 [C4].

5. **대금 지급·계약 구조가 실질 거래와 맞지 않는 경우**
   - 대금이 명의상 거래처가 아니라 다른 사람에게 지급되거나, 계약서상 명의와 실제 거래 주체가 어긋나면 허위거래 정황으로 보았습니다.  
   - 예: 개인 명의로 양수·인수된 정황, 실질 대표자로 볼 근거가 없는 경우 [C3].

### 반복되는 증거 유형
- **세금계산서**, **사업자등록증**, **출하전표**, **계좌이체**, **거래명세표**, **확인서**, **통장사본** 등이 자주 등장합니다.
- 다만 이런 자료가 있다고 해서 자동으로 공제가 인정되는 것은 아니고,  
  **실제 공급자 확인**, **실물거래 존재**, **거래처의 정상 영업 여부**까지 종합해서 판단했습니다. [C1][C4][C5]

### outcome 경향
- 전체적으로는 **기각이 압도적**입니다.  
  - 기각 61, 인용 13, 일부인용 4, 재조사 2
- 즉, **사실과 다른 세금계산서가 인정되면 매입세액 불공제가 원칙적으로 유지**되는 경향이 강합니다. [C2][C3][C4]
- 다만 예외적으로, 청구인이 **실제 거래라고 믿을 만한 객관적 사정**과 **주의의무 이행**을 충분히 입증하면 인용됩니다.  
  - 예: 거래 전 서류 확인, 출하전표·운송확인서 확인, 법인계좌 송금 등을 통해 선의가 인정된 경우 [C5]  
  - 실제 공급자가 달랐더라도 사전에 알기 어려웠다고 보아 공제를 인정한 경우 [C1]

요약하면, 이 그래프에서는 **“세금계산서의 형식”보다 “실제 공급자와 실물거래의 일치 여부”가 핵심**이고, 그 불일치가 확인되면 **매입세액 공제는 대체로 부인**되며, **선의·주의의무 입증이 있을 때만 예외적으로 인정**되는 패턴이 반복됩니다.


---


[(1,
  '## Retrieved Chunks\n1. score=16.712 file=조심-2019-전-0283.pdf chunk=4\n   법인이 쟁점세금계산서를 뒤늦게 교부 받은 것에 아무런 귀책사유가 없으며 또한 ,  기에는 세금을 탈루하겠다는 등의 어떠한 불법적인 의도도 없었다.  라 따라서 대법원 판결의 별개의견과 같이 세금계산서의 기재사항에 따라 거래사    ( )  ,  이 확인되고 부가가치세의 거래징수도 정상적으로 이루어졌으나 납세의무자의 탓으로  리기 어려운 특별한 사정으로 인하여 그 거래시기가 속하는 과세기간 내\n2. score=14.147 file=조심-2019-전-0283.pdf chunk=2\n   거래에 대한 매출세액이 누락된 사실     3) OOO 발견한 후 수정신고를 통하여 이를 바로잡고자 하였다 이 과정에서  담당직원은 수 .  OOO 신고를 위해 전자세금계산서를 다시 발급하여야 하는 것으로 잘못 알고, 2018.2.19.  건 거래의 공급가액만큼 증액하여 전자세금계산서를 발급하였다가  이를 다 2018.2.21.  취소하였다 그런데 이는 순전히 대지금속 담당직원이 종이세금계산\n3. score=13.260 file=조심-2015-부-3287.pdf chunk=6\n   의  예금계좌로      3)  OOO OOO OOO  체받은 후 즉시 현금으로 출금하여 동 대금의 사용처가 불분명한 자료상의 전형적인 거 ,  형태를 보인다는 의견이나,  청구인은 쟁점거래시  의 요구에 따라 고철을 납품받고 대금을  명의의 예        OOO OOO  계좌로 이체대금의  하였고일부 금액은 현금으로 지급하였을 뿐이고 이 쟁점거 ( 55%) ,  , OOO 대금을 어떻게 \n4. score=13.064 file=조심-2012-구-1315.pdf chunk=2\n   한 것으로 납품처인  가  OOO 청구법인에게 고철을 직접 운반하는 특수성 등을 고려할 때세금계산서는  ( OOO→ 청구법인 관련 매입세금계산서가 사실과 다른 세금계산서에 해

### 6-2. 상담형 질문 (파이프라인 2의 issue 구조화가 유리한 유형)

법률 용어 대신 상황을 풀어서 묻기 때문에, 의미 기반으로 쟁점을 잡는 쪽이 유리합니다.

In [ ]:
results = compare("실제 거래는 존재했으나 거래 상대방의 명의만 바뀐 경우, 심판원은 보통 어떤 논리로 과세관청의 처분이 정당하다고 판단해?")

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Issue-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...


### 질문

> 실제 거래는 존재했으나 거래 상대방의 명의만 바뀐 경우, 심판원은 보통 어떤 논리로 과세관청의 처분이 정당하다고 판단해?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _38.3s_

제공된 검색 결과만 보면, 심판원은 **“실물거래가 있었는지”만 보지 않고, 세금계산서상의 거래상대방이 실제 거래상대방과 일치하는지**를 핵심으로 봅니다.  
즉, **실제 물품은 오갔더라도 대표자 이름만 바뀌어 세금계산서상의 명의가 실제와 다르면**, 그 세금계산서를 **사실과 다른 세금계산서**로 보거나, **상대방이 명의대여·자료상인지**를 따져 과세관청 처분을 정당하다고 판단하는 흐름입니다[1][3][7][8].

구체적으로는 다음 논리가 반복됩니다.

1. **거래상대방이 정상사업자인지, 명의대여인지 확인해야 한다**  
   심판원은 거래 당시 사업자등록증, 통장사본 등을 받았는지, 실제로 상대방이 정상업체인지 확인했는지를 봅니다. 상대방이 **“명의만 대여한 것”**으로 확인되면, 실물 일부가 있었다고 해도 문제 삼습니다[1][7].

2. **실물거래가 있어도 세금계산서의 상대방 명의가 다르면 문제될 수 있다**  
   결과 [1]에는 실제 구리가 입고되었더라도 **세금계산서상의 대표자 이름과 사업자등록증상의 이름이 달라** 재작성·정정한 사정이 보입니다. 이런 경우 심판원은 형식상 명의가 맞지 않는 점을 중시합니다[1].

3. **자료상으로 확정된 업체와의 거래면 더욱 처분을 인정한다**  
   결과 [3], [7], [8]처럼 상대방이 **실물거래 없이 세금계산서만 발행한 자료상**이거나, 실제 거래 없이 형식상 거래를 한 것으로 보이면, 심판원은 과세관청 판단을 유지하는 경향이 있습니다[3][7][8].

정리하면, 질문하신 경우 심판원은 보통  
**“실제 거래의 외형이 일부 있어도, 세금계산서상의 거래명의가 실제 거래주체와 달라지고 그 상대방이 명의대여자·자료상으로 확인되면, 실질과 다른 세금계산서이므로 과세처분은 정당하다”**  
는 논리로 봅니다[1][3][7][8].

원하시면 제가 이걸 **“한 줄 요약”**이나 **“심판례 문장 스타일”**로도 다시 정리해드릴게요.


---

### ✅ 파이프라인 2 · Issue-based (structural)  · _27.1s_

심판원은 보통 **“실제 거래가 있었는지”만으로는 부족하고, 세금계산서의 공급자 명의가 실제 공급자와 다른 경우에는 형식보다 실질을 보되, 그 명의위장 사실을 알지 못했고 알지 못한 데 과실이 없다는 점을 납세자가 입증해야 한다**는 논리로 과세관청 처분을 정당하다고 보는 경향이 있습니다[1].

구체적으로는:

1. **실제 공급자와 세금계산서상 공급자가 다르면**
   - 원칙적으로 매입세액 공제를 부인할 수 있고,
   - **선의의 거래당사자**인지 여부를 따집니다[1].

2. **입증책임은 납세자에게 있음**
   - 심판원은 납세자가 그 명의위장 사실을 몰랐고, 몰랐던 데 과실도 없다는 점을 **객관적으로 입증해야 한다**고 봅니다[1].

3. **기본적인 확인의무를 다했는지 봄**
   - 사업장 방문, 실질 운영자 확인, 거래관행에 맞는 확인 등 기본적인 주의의무를 하지 않았다면,
   - 실제 거래가 있었다고 하더라도 **선의의 거래당사자로 보기 어렵다**고 판단합니다[1].

4. **명의만 빌려준 경우도 실질을 따짐**
   - 사업자등록, 신고, 임차관계, 실제 운영 여부 등을 종합해 **형식이 아니라 실질상 누가 사업자였는지**를 판단합니다[4].

즉, 질문하신 상황에서는 심판원이 대체로  
**“실제 거래 자체가 있었더라도, 거래 상대방 명의가 바뀐 것이 명의위장이라면 납세자가 이를 몰랐고 과실이 없다는 점을 입증하지 못하는 한 과세처분은 정당하다”**  
는 구조로 판단한다고 보면 됩니다[1][4].

원하시면 이 논리를 **답변서/의견서 스타일**로 3줄 정도로 더 압축해 드릴게요.


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _41.4s_

제공된 그래프 컨텍스트만 보면, **“실제 거래가 있었다”는 주장 자체보다, 거래 상대방의 명의가 바뀌었음에도 그 변경 사실이 반영되지 않은 서류와 객관증빙의 부실**이 있으면 심판원은 대체로 **과세관청 처분을 정당**하다고 보는 경향이 강합니다.

## 반복되는 판단 논리

1. **거래상대방이 자료상으로 보이거나 실제 공급이 의심되는 정황**
   - 상대방이 자료상으로 고발되었거나, 신고가 없고, 실제 입출고 사실이 확인되지 않으면 실제거래를 인정하지 않는 방향으로 갑니다.  
   - 대표적으로 거래상대방이 전부 자료상으로 고발되었고 실제 유류 입출고도 없었다는 점이 핵심 근거였습니다. [C4]

2. **명의 변경 후에도 세금계산서·거래명세서·출하전표 등에 ‘변경 전 상호/소재지’가 기재된 경우**
   - 이 패턴은 “실제 거래가 있었다”는 주장과 별개로, **서류의 신빙성이 약하다**는 쪽으로 작용합니다.  
   - 즉, 실제 거래를 가장한 명의형식만 남아 있다고 보거나, 최소한 정상적인 거래증빙으로 보기 어렵다고 판단하는 흐름입니다. [C4]

3. **객관적 증빙 부족**
   - 청구인이 “실제 거래였다”고만 말하고, 금융증빙·계좌흐름·출하전표의 정상성 등을 뒷받침할 자료를 못 내면 기각으로 가는 경우가 반복됩니다. [C1][C2][C4]
   - 특히 구체적이고 객관적인 증빙이 없으면, 실제거래 주장만으로는 부족하다고 봅니다. [C1][C2]

4. **선의의 거래당사자 주장도 엄격**
   - 상대방 명의가 바뀌었더라도, 서류상 변경 내용이 맞지 않거나 전표가 부실하면 “주의의무를 다했다”는 주장도 잘 받아들여지지 않습니다. [C4]

## outcome 경향

- 이 유형은 전체적으로 **기각이 다수**입니다.  
  - issue_count 17건 중 raw_outcomes 기준으로 **기각 10건**이 가장 많습니다.
- 반대로 **실제 거래로 인정되어 청구가 인용되는 경우**는 상대적으로 적고,  
  - 본문 대표사건에서도 실질적으로는 **객관증빙 부족 / 형식적 서류 불일치**가 있으면 과세관청 처분을 유지했습니다. [C1][C2][C4]

## 요약하면

심판원은 보통  
- **실제 공급이 있었는지**를 먼저 보되,  
- **거래상대방 명의 변경이 서류에 반영되지 않았거나**,  
- **세금계산서·거래명세서·출하전표가 부실하고**,  
- **금융흐름 등 객관증빙이 없으면**  

“실제 거래” 주장을 받아들이지 않고 **가공 또는 사실과 다른 거래로 보아 과세관청 처분을 정당**하다고 판단하는 경향이 있습니다. [C1][C2][C4]


---


[(1,
  '## Retrieved Chunks\n1. score=17.191 file=조심-2011-중-1186.pdf chunk=9\n   ○○ ○○○ 사를 하여 자료상확정자로 고발된 업체입니다 당시 조사보고서를 보면 는 사업장 .  , ○○○ 도 없고 당시의 대표자는 명의만 대여한 것으로 되어 있는데 귀하는 이러한 사실을 알았 습니까? 문 전혀 몰랐습니다 급 구리는 고가의 품목이라 조 이 본인에게 매입의사를 타 ( )  . A ○○○ 진할 때도 실제 매입할 구리가 있는지가 중요한 것이었지 어느 업체의 구리인지는 중요 한 것이 \n2. score=14.674 file=국심-2000-서-2316.pdf chunk=8\n    부가가치세 과세대상이 되는지 여부에 대해서는 어떠한 안내도 받지 못하였던 바,  처분청이 쟁점전화서비스용역을 부가가치세 과세대상이 아니라고 하여 이 건 환급하지  않은 처분은 신의성실의 원칙에 반하는 것이라고 주장한다.  나 일반적으로 조세관계에서 과세관청의 행위에 대하여 신의성실의 원칙이 적용되는 요 ( )  건으로는 첫째과세관청이 납세자에게 신뢰의 대상이 되는 공적인 견해표명을 하여야 \n3. score=13.924 file=조심-2013-중-0993.pdf chunk=4\n   확인서 유류대금입금거래내역조회서 등을 제출하였다 ,  . 가 청구법인의 주유소는 임대가 아닌 법인의 자산인 까닭에 모든 일에 신중 ( )  하였고 그 결과 거래마다 철저한 확인을 거쳤는바 거래 시작 전 사업자등록증 ,  , 1)  ,  석유판매업등록증 사업자명의의 통장사본 탱크가 없을 경우 탱크임대차계약서를  ,  ,  제출받았고 운전기사가 가져 온 출하전표를 받은 후 전표번호 수송장비번호 \n4. score=12.716 file=조심-2025-서-2015.pdf chunk=3\n    따라 하도급을 이용하여 시공하는 방법을 선택하였는바 하도급 구조 ,  와 관련된 수많은 사업체를 직접 관리한다면 적지 않은 비용이 소모되므로 하도급 구조에서  ,  용역의 

### 6-3. 패턴형 질문 (파이프라인 3 Graph RAG가 필요한 유형)

개별 사건 몇 개가 아니라, 500건 전체에서 결과가 갈리는 패턴을 비교하는 질문입니다.
이런 질문은 `graph_mode="auto"`가 자동으로 `outcome-comparison` 모드를 고릅니다.

In [ ]:
results = compare("세무조사 나와서 억울하게 엮인 부가가치세 심판 사건들 보면, 보통 법원에서 어떤 증거를 가장 중요하게 봐줘?")

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Issue-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...


### 질문

> 세무조사 나와서 억울하게 엮인 부가가치세 심판 사건들 보면, 보통 법원에서 어떤 증거를 가장 중요하게 봐줘?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _26.7s_

위 검색 결과만 보면, 부가가치세 심판·소송에서 **가장 중요하게 보는 증거는 “실제 거래가 있었다는 객관적 증빙”**입니다. 특히 다음 자료들이 핵심으로 보입니다.

1. **정산서와 자금흐름이 맞는지**
   - 정산서는 “가장 중요한 최종 증빙”으로 언급되고, 세금계산서 수수내역 및 금융증빙으로 실제 입출금이 확인되면 거래 실재를 뒷받침한다고 봤습니다.[1]
   - 부가가치세 신고자료, 송금내역, 구매내역, 정산서가 거래흐름을 명확히 보여준다고 한 사례도 있습니다.[1]

2. **금융증빙(계좌이체, 입출금 내역 등)**
   - 실제 대금지급증빙이 중요하고, 객관적인 금융거래 내역이 있어야 실제 거래를 인정받기 쉽습니다.[1][4]
   - 반대로 금융거래 내역만으로도 거래 실체가 곧바로 인정되는 것은 아니지만, 적어도 실재를 뒷받침하는 핵심 자료로 다뤄집니다.[8]

3. **세금계산서와 거래내용의 일치**
   - 세금계산서 수수내역이 정산서, 금융증빙과 일치해야 합니다.[1]
   - 사실과 다른 세금계산서인지 여부는 처분청이 직접 증거로 입증해야 한다는 취지의 언급도 있습니다.[1]

4. **구체적인 거래내역**
   - 상품 종류, 수량, 단가, 거래명세서 같은 구체적 자료가 중요합니다.[2]
   - 이런 구체 자료 없이 단순한 금전거래만으로 매입누락이나 가공거래를 인정하는 것은 부당하다는 취지의 주장도 나옵니다.[2]

5. **단순 계약서만으로는 부족할 수 있음**
   - 계약서와 정산서만으로는 용역이나 거래의 실체를 확인하기 어렵다고 본 사례가 있습니다.[8]
   - 즉, 서류 형식보다 **실제 수행 내용과 돈의 흐름이 맞는지**를 더 중시합니다.[8]

정리하면, 이 검색 결과들만 기준으로는 법원·심판원은 보통 **“계약서 자체”보다도 “정산서 + 금융증빙 + 세금계산서 + 구체적 거래내역”의 일치 여부**를 가장 중요하게 봅니다.[1][2][8]

근거가 부족한 부분까지 일반화하면 안 되므로, **이 검색결과만으로는 사건 유형별로 무엇을 ‘항상’ 가장 중시한다고 단정하기는 어렵습니다**. 다만 제시된 사례들에서는 위의 객관적 증빙이 반복해서 핵심으로 나타납니다.[1][4][8]


---

### ✅ 파이프라인 2 · Issue-based (structural)  · _22.7s_

제공된 사례들만 보면, 부가가치세 심판에서 가장 중요하게 보는 증거는 **“실제 거래가 있었다는 객관적 증빙”**입니다. 특히 다음이 핵심으로 보입니다.

1. **대금 지급의 객관적 증빙**
   - 무통장입금증, 계좌이체 내역 등 실제로 돈이 오갔는지 보여주는 자료가 중요합니다.[4]
   - 다만 **송금내역만으로는 부족**할 수 있고, 거래 실재를 함께 뒷받침해야 합니다.[1][4]

2. **물품의 실제 입출고·거래내역**
   - 매입품목과 수량, 거래명세서, 거래처원장, 재고수불현황, 매출상세내역 같은 자료가 중요합니다.[1]
   - 주류 사건에서도 **매입·매출 거래명세서, 거래처원장, 재고수량 확인자료**를 제시하지 못한 점이 불리하게 작용했습니다.[3]

3. **거래상대방의 실재 및 조사결과**
   - 상대방이 세무조사에서 **자료상(실물거래 없이 세금계산서만 유통)**으로 확정되면, 납세자가 제출한 자료가 더 엄격하게 보입니다.[1]
   - 이 경우 제출 증빙이 **위장 증빙**인지까지 판단합니다.[1]

4. **객관성·일관성이 있는 전체 정황**
   - 단순히 “억울하다”는 주장이나 내부적으로 만든 자료만으로는 부족합니다.[1][4]
   - 심판부는 실제 사용액·실제 거래액처럼 **추정이 아니라 실재를 보여주는 자료**를 선호합니다.[5]

정리하면, 이 유형 사건에서는 법원이 보통  
**“실제 돈이 갔는지” + “실제로 물건이 갔는지” + “그걸 뒷받침하는 장부·재고·거래명세가 있는지”**를 가장 중요하게 봅니다.[1][3][4]

반대로, **송금내역만 있거나**, **사후에 맞춘 듯한 자료**, **실제 입출고를 확인할 수 없는 장부**만 있으면 인정받기 어렵다는 흐름입니다.[1][4]

원하시면 제가 이 검색결과 기준으로  
**“세무조사 때 챙겨야 할 증빙 체크리스트”** 형태로도 정리해드릴게요.


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _33.0s_

세무조사로 엮인 부가가치세 심판례들을 보면, 법원이 가장 중요하게 보는 건 **“세무조사 자체의 하자”보다도 결국 과세근거를 뒷받침하는 객관적 증거가 있느냐**입니다. 반복 패턴상 특히 아래 증거들이 핵심입니다.

1. **세금계산서·거래명세표·계약서·이메일 같은 거래증빙**
   - 실물거래 여부, 주문 내용, 품목·수량·단가, 납품 시점이 서로 맞는지가 중요합니다.
   - 예: 이메일에 주문 내용과 단가·수량이 구체적으로 적혀 있고, 퀵서비스 인수일과 납품일이 맞아떨어져 실물거래를 인정한 사례가 있습니다. 반대로 처분청이 “가공거래”라고 주장해도 그걸 뒤집을 객관적 증거가 부족하면 인용됩니다. [C3]

2. **금융자료(계좌이체내역, 통장, 현금결제 흐름)**
   - 돈이 어떻게 오갔는지, 매입대금·매출대금의 흐름이 거래 실체와 맞는지가 매우 중요합니다.
   - 예: 계좌 송금, 즉시 현금결제, 외상매출금 회수 여부 같은 자금흐름이 거래 실재 판단의 핵심 근거로 작동했습니다. [C3], [C4]

3. **세무조사 과정에서 나온 내부 자료·조사자료**
   - 세무조사 복명서, 조사자료, 조사결과 자체는 처분의 출발점이지만, 그 자체만으로는 부족하고 **추가적인 객관적 증빙**이 필요합니다.
   - 예: 재조사 금지 주장 사건에서는 “별도의 세무조사가 있었다”는 객관적 증빙이 없어서 처분청이 이겼습니다. 즉, 조사 이후 내부 검토나 사실 확인 정도만으로는 재조사로 보기 어렵다고 봤습니다. [C2]
   - 절차 하자 주장 사건에서도 관리책임자 입회, 통지서 교부, 장부·통장 복사, 보관증을 통한 예치 같은 기록이 있으면 절차 위법 주장을 쉽게 받아들이지 않았습니다. [C1]

4. **청구인 측의 반증자료**
   - 거래처원장, 재고수량 확인자료, 매입·매출명세, 부가가치세 정상신고, 고소장·불기소/무혐의 자료 등이 있으면 납세자에게 유리하게 작용할 수 있습니다.
   - 다만 이런 자료도 **거래 실체를 직접 입증**해야 하고, 단순 주장만으로는 부족합니다. [C3], [C4]

정리하면, 이 유형의 사건에서 법원·심판부가 가장 중요하게 보는 증거는 **세금계산서 같은 형식자료 단독이 아니라, 그 거래를 실제로 뒷받침하는 거래명세·계약·이메일·운송기록·계좌흐름 같은 객관적 증거**입니다.  
반대로 **“세무조사가 있었다”는 사실 자체나 조사관의 추정만으로는 부족**하고, **객관적 증빙 미제시**가 반복적으로 기각 사유로 나타납니다. [C2], [C4]

전체 outcome 경향도 이와 같습니다.  
- 전체 80건 중 **기각 63건**이어서, **납세자 주장이 받아들여지지 않는 경우가 훨씬 많습니다.**
- 다만 **인용 11건**도 있고, 인용된 사건들은 대체로 **실물거래 존재를 뒷받침하는 구체적 자료가 충분하거나**, 처분청의 가공거래·재조사 주장에 **객관적 뒷받침이 부족한 경우**였습니다. [C3]
- 따라서 반복되는 그래프 패턴은 **“조사상 하자 주장보다 실질거래 입증이 핵심”**, 그리고 **“객관적 증빙이 없으면 과세가 유지되고, 있으면 처분이 뒤집힐 수 있음”**으로 요약됩니다.


---


[(1,
  '## Retrieved Chunks\n1. score=12.379 file=조심-2025-구-0998.pdf chunk=3\n   하여 정산하고 세금계산서를 수취하고 다 결국 당사자간 수수료를 확정함에 있어 정산서는 가장 중요한 최종 증빙으로서 해당 정산서는 세금계 .  서 수수내역와 일치하며관련 금융증빙을 통하여 실제 입출금내역을 확인할 수 있다 ,  . 나    ( ) 국내 면세점에서 상품을 구매하는 따이공은 어느 특정여행사에 계속하여 종속되는 가이드나 직 신분이 아니다 이들 따이공은 가이드와의 친분이나 구매대행업을\n2. score=12.015 file=조심-2017-구-0609.pdf chunk=1\n   7.1.13.  제기하였다. 청구인 주장 및 처분청 의견 2.  가 청구인 주장 .       처분청은 쟁점사업장의 총 매입대금쟁점계좌로 송금한 금액어음결제액주류 (1)  ( + + 구매카드 결제액에서 쟁점매입처와 주식회사  이하  라 한다로부터 수 ) OOO( “OOO” ) 취한 매입세금계산서 공급대가를 차감하여 매입누락액을 산정하였는바 년 제 , 2011 기부터  년 제기분 매입누락액을 \n3. score=11.363 file=조심-2019-구-4113.pdf chunk=6\n    같다에게 조사를 시작하기  일 전에 조사 .  ) 15 상 세목조사기간 및 조사 사유 그 밖에 대통령령으로 정하는 사항을 통지이하 이 조에 ,  ,  ( 사전통지라 한다하여야 한다 다만사전통지를 하면 증거인멸 등으로 조사 목적을 달성 " " ) .  ,  수 없다고 인정되는 경우에는 그러하지 아니하다.  [page 8] 제 조의 세무조사 기간 세무공무원은 조사대상 세목ㆍ업종ㆍ규모조사 난이도\n4. score=9.487 file=국심-2003-서-3244.pdf chunk=3\n      ,  소명시 가공매입으로 소명하였다가 심판청구시에는 추가 항변자료를 제출하여 입증자료의 부족으로 억울하게 가공 매입으로 부과되었다고 주장만 할 뿐 물건을 실지 매입한  무

### 6-4. retrieval log까지 함께 보기

각 파이프라인이 실제로 무엇을 검색했는지(chunk / issue / graph 대표사건)를 같이 확인하고 싶을 때.

#### 6-4-1. 키워드형 질문

In [10]:
results1 = compare("사실과 다른 세금계산서가 문제된 사건들에서 매입세액 공제가 부인된 이유는 뭐야?", show_logs=True)


▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Issue-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...


### 질문

> 사실과 다른 세금계산서가 문제된 사건들에서 매입세액 공제가 부인된 이유는 뭐야?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _17.7s_

사실과 다른 세금계산서 사건에서 매입세액 공제가 부인된 이유는, **세금계산서가 실제 거래와 맞지 않거나 법정 발급기한을 넘겼다고 보았기 때문**입니다.

구체적으로는 다음과 같습니다.

1. **공급시기가 속한 과세기간·신고기간을 넘겨 발급받은 경우**
   - 공급시기 이후, 그리고 신고기간까지 지나서 세금계산서를 받은 경우에는 「부가가치세법」상 **세금계산서를 발급받지 아니한 경우**로 보아 매입세액 공제를 인정하지 않았습니다. [1]

2. **필요적 기재사항이 실제 거래와 불일치한 경우**
   - 세금계산서의 **공급자, 공급받는 자, 가액, 시기** 등이 실제 거래와 다르면 “사실과 다른 세금계산서”로 보았습니다. [2]
   - 특히 실제 공급주체와 세금계산서상의 주체가 다르거나, 공사 시행주체가 아닌 다른 사업자로부터 받은 세금계산서도 문제되었습니다. [6]
   - 공사대금을 지급한 상대방과 세금계산서 교부 주체가 맞지 않는 경우에도 사실과 다른 세금계산서로 보아 불공제했습니다. [8]

3. **실질적인 재화·용역 공급이 없는 가공거래로 본 경우**
   - 명목상 거래이거나 실제 공급 없이 자료상·폭탄업체 등을 끼운 거래라고 판단되면, 그 과정에서 받은 세금계산서는 매입세액 공제 대상이 아니라고 보았습니다. [4][5]

즉, 핵심은 **“실제 거래와 세금계산서가 일치하지 않거나, 거래 자체가 허위·가공으로 판단되면 매입세액 공제가 부인된다”**는 점입니다. [2][4][5][6][8]

원하시면 제가 이걸 **사실과 다른 세금계산서 유형별로 표로 정리**해드릴게요.

<details><summary>retrieval log 보기</summary>

```
## Retrieved Chunks
1. score=16.712 file=조심-2019-전-0283.pdf chunk=4
   법인이 쟁점세금계산서를 뒤늦게 교부 받은 것에 아무런 귀책사유가 없으며 또한 ,  기에는 세금을 탈루하겠다는 등의 어떠한 불법적인 의도도 없었다.  라 따라서 대법원 판결의 별개의견과 같이 세금계산서의 기재사항에 따라 거래사    ( )  ,  이 확인되고 부가가치세의 거래징수도 정상적으로 이루어졌으나 납세의무자의 탓으로  리기 어려운 특별한 사정으로 인하여 그 거래시기가 속하는 과세기간 내
2. score=14.147 file=조심-2019-전-0283.pdf chunk=2
   거래에 대한 매출세액이 누락된 사실     3) OOO 발견한 후 수정신고를 통하여 이를 바로잡고자 하였다 이 과정에서  담당직원은 수 .  OOO 신고를 위해 전자세금계산서를 다시 발급하여야 하는 것으로 잘못 알고, 2018.2.19.  건 거래의 공급가액만큼 증액하여 전자세금계산서를 발급하였다가  이를 다 2018.2.21.  취소하였다 그런데 이는 순전히 대지금속 담당직원이 종이세금계산
3. score=13.260 file=조심-2015-부-3287.pdf chunk=6
   의  예금계좌로      3)  OOO OOO OOO  체받은 후 즉시 현금으로 출금하여 동 대금의 사용처가 불분명한 자료상의 전형적인 거 ,  형태를 보인다는 의견이나,  청구인은 쟁점거래시  의 요구에 따라 고철을 납품받고 대금을  명의의 예        OOO OOO  계좌로 이체대금의  하였고일부 금액은 현금으로 지급하였을 뿐이고 이 쟁점거 ( 55%) ,  , OOO 대금을 어떻게 
4. score=13.064 file=조심-2012-구-1315.pdf chunk=2
   한 것으로 납품처인  가  OOO 청구법인에게 고철을 직접 운반하는 특수성 등을 고려할 때세금계산서는  ( OOO→ 청구법인 관련 매입세금계산서가 사실과 다른 세금계산서에 해당된다는 사실을  ),  알지 못한데 대한 과실이 없는 선의의 거래당사자로  를 보호하기 위하여 계속  OOO 거래자로서 정상거래임에도 현재 폐업하고 매출이 가장 적은  에 관하여는 사 OOO 실과 다르게 허위의 진술을 
5. score=12.858 file=조심-2010-서-1708.pdf chunk=1
   6 2 원 년 제기  원을 경정고지하였다 2,176,964,300 , 2007 1 482,130,280 · . 다청구법인은 이에 불복하여  심판청구를 제기하였다 .  2010.3.30.  . 청구인 주장 및 처분청 의견 2.  가청구인 주장 .  처분청은 부가가치세를 납부하지 아니한 업체를 거친 재화를 매입거래처가 매입한 후  청구법인에게 매출하였다 하여 쟁점세금계산서를사실과 다른 세금계산서
6. score=12.789 file=조심-2020-중-2393.pdf chunk=0
   [page 1] 문서번호 조심 중 -2020- -2393 결정유형 기각 세목 부가가치세 생산일자 2020. 12. 21. 귀속연도 2019 제목 사실과 다른 세금계산서는 매입세액 공제에 해당하지 아니함 요지 청구인은 쟁점건물의 사용승인일 이후에 잔금을 지급 및 소유권이전을 하였 고 건물공사 시행주체가 아닌 다른 사업자로부터 받은 세금계산서는 사실 ,  과 다른 세금계산서에 해당함 내용 결정내
7. score=12.704 file=조심-2017-부-4938.pdf chunk=26
    전체 매출액의  이 70%  이었고 의 경우 쟁점거래로 청구법인과의 거래물량이 증가한  , OOO 1 것으로 나타나는 청구법인이  와 쟁점거래를 한 목적이 조세의 회피 또 OOO 1 는 탈루의 의도라기보다는OOO 의 자동차부품을 생산하는 계열사인  가 철판 고무 등 원자재의 안정적인 확보를 OOO ,  해  그룹의 경쟁력 향상 및 거래당사자인 청구법인과  의 사업의 계속성  OOO OOO 
8. score=12.676 file=국심-1999-전-1288.pdf chunk=1
   접지급하여 쟁 ,  ,  점건물을 직접 신축하였다고 인정되며 세금계산서도  종합건설에 공사대금을 지 ,  ○○○ 급하였다고 하는 때에 교부되지 않아 쟁점세금계산서를 사실과 다른 세금계산서로 보 아 매입세액 불공제하여 과세한 당초처분이 정당하다.  심리 및 판단  3.  가 쟁점  .  쟁점세금계산서를 사실과 다른 세금계산서로 보아 매입세액불공제한 처분의 당부를 가 리는데 있다.  [page 
9. score=12.489 file=조심-2019-전-0283.pdf chunk=3
   ,  ,  .  따라서 쟁점세금계산서는 필요적 기재사항이 사실과 다르게 적힌 경우라고 할      5)  ,  없으므로 부가가치세법 제 조 제항 제호가 적용될 수 없다 그럼에도 처분청은  , 39 1 2 .  「 」 호를 적용하여 청구법인의 경정청구를 거부하였는바 이 건 거부처분은 위법하므로 취 ,  되어야 한다.     쟁점세금계산서가 매입세액이 공제되지 않는 경우를 정한 법령에 문언상 해
10. score=11.926 file=조심-2020-구-7973.pdf chunk=2
   공급자가 시 」 설을 공급하면서 거래징수한 부가가치세에 대하여 시설대여업자는 매입세액을 공제받 지 못하고 이후 시설대여업자가 시설 임차인에게 시설을 임대하면서 자신이 공제받지  ,  못한 매입세액 상당액을 매월 임대료에 포함시키는 방식으로 전가시킴으로써 결국 해 당 시설을 자신의 과세사업에 사용하는 임차인이 해당 시설에 대한 매입세액을 공제받 지 못하는 불합리한 결과를 해소하기 위한 것이다.
```
</details>


---

### ✅ 파이프라인 2 · Issue-based (structural)  · _17.8s_

사실과 다른 세금계산서 사건들에서 매입세액 공제가 부인된 핵심 이유는, **세금계산서상 공급자와 실제 공급자가 다르거나, 공급시기와 맞지 않는 세금계산서를 수취했기 때문**입니다. 즉, 세금계산서가 실질 거래를 정확히 반영하지 않으면 부가가치세법상 매입세액 공제 대상이 아니라고 본 것입니다 [1][2][4].

구체적으로 보면:

1. **실제 공급자와 세금계산서상 공급자가 다른 경우**
   - 어떤 사건에서는 건물 공사의 실제 시행주체가 다른 사업자로 보이는데, 세금계산서는 그와 다른 사업자 명의로 발급되어 있었습니다. 이 경우 “사실과 다른 세금계산서”에 해당한다고 보아 매입세액 공제가 부인되었습니다 [1].
   - 유류 거래 사건에서도 거래처가 실제로 유류를 거래한 사실 없이 자료상으로 확인되어, 형식상 세금계산서와 실제 거래가 일치하지 않는다고 판단했습니다 [4].

2. **공급시기와 맞지 않는 세금계산서를 수취한 경우**
   - 건물 사용승인일이 공급시기인데, 세금계산서를 더 늦게 받은 경우 그 세금계산서는 적법한 매입세액 공제 요건을 충족하지 못한다고 보았습니다 [2].
   - 특히 간이과세자에서 일반과세자로 전환된 경우 과세기간이 나뉘므로, 사용승인일과 세금계산서 수취일이 같은 달에 있어도 같은 과세기간이 아니면 공제가 안 된다고 판단했습니다 [2].

3. **거래 실재성이나 공급자 확인이 부족한 경우**
   - 거래 상대방이 자료상으로 드러났거나, 실제 사업 운영 흔적이 부족한 경우 매입세액 공제가 부인되었습니다 [4][5].
   - 사업자등록증, 계좌이체, 출하전표 같은 형식적 증빙만으로는 실제 공급자 여부나 정상 거래를 충분히 입증하지 못한다고 본 사례도 있습니다 [4][5].

4. **선의의 거래당사자라고 보기 어려운 경우**
   - 세금계산서가 사실과 다르더라도, 납세자가 몰랐고 과실도 없었다면 다툼이 될 수 있지만, 여러 사건에서는 거래처 실사나 물량 출처, 사업장 실재성 등을 충분히 확인하지 않아 선의·무과실이 인정되지 않았습니다 [4][5].
   - 반대로, 선의 여부가 충분히 확정되지 않으면 재조사가 필요하다고 본 사례도 있습니다 [3].

정리하면, 이들 사건에서 매입세액 공제가 부인된 이유는 **세금계산서가 실제 거래의 공급자·공급시기·거래 실체를 제대로 반영하지 못했기 때문**이고, 그에 더해 **납세자가 실제 공급자를 확인하기 위한 주의의무를 다하지 못했다고 본 경우**가 많았습니다 [1][2][4][5].

원하시면 제가 이 내용을 **“사실과 다른 세금계산서” 판단 기준** 형태로 3줄 요약해드릴게요.

<details><summary>retrieval log 보기</summary>

```
## Retrieved Issues
1. score=14.912 file=조심-2020-중-2393.pdf issue=0
   쟁점: 쟁점세금계산서가 실제 공급자와 세금계산서상 공급자가 다른 사실과 다른 세금계산서에 해당하여 매입세액 공제가 배제되는지 여부
   결론: 쟁점세금계산서는 사실과 다른 세금계산서에 해당하므로 관련 매입세액은 공제되지 않으며, 경정청구 거부처분은 정당하다고 보아 청구를 기각하였다.
2. score=11.360 file=국심-2005-전-3976.pdf issue=1
   쟁점: 건물 사용승인일과 세금계산서 수취일이 같은 과세기간에 속하여 매입세액 공제가 가능한지 여부
   결론: 기각. 사용승인일과 세금계산서 수취일은 동일한 과세기간에 속하지 않으므로 매입세액 공제 대상이 아니다.
3. score=11.223 file=조심-2025-인-0392.pdf issue=1
   쟁점: 청구인이 쟁점세금계산서 수수에 관하여 선의·무과실의 거래당사자에 해당하는지 여부
   결론: 선의·무과실 여부를 현 단계에서 확정하지 않고, 재조사 대상으로 보아 판단을 유보한 일부 인용.
4. score=10.977 file=조심-2013-부-2294.pdf issue=0
   쟁점: 쟁점매입처로부터 수취한 쟁점세금계산서를 사실과 다른 세금계산서로 보아 매입세액을 불공제한 처분이 정당한지 여부
   결론: 기각. 쟁점세금계산서는 사실과 다른 세금계산서로 보아야 하므로 매입세액 불공제 및 과세한 처분이 정당하다.
5. score=10.519 file=조심-2012-구-1315.pdf issue=1
   쟁점: 청구법인이 명의위장 또는 자료상 거래 여부를 알지 못한 선의의 거래당사자로서 관련 매입세액 공제를 받을 수 있는지 여부
   결론: 기각, 선의의 거래당사자에 해당하지 않아 관련 매입세액 공제가 부인됨

## Added Raw Chunks
R1. score=0.000 file=조심-2020-중-2393.pdf chunk=0
    [page 1] 문서번호 조심 중 -2020- -2393 결정유형 기각 세목 부가가치세 생산일자 2020. 12. 21. 귀속연도 2019 제목 사실과 다른 세금계산서는 매입세액 공제에 해당하지 아니함 요지 청구인은 쟁점건물의 사용승인일 이후에 잔금을 지급 및 소유권이전을 하였 고 건물공사 시행주체가 아닌 다른 사업자로부
R2. score=0.000 file=조심-2020-중-2393.pdf chunk=1
    매입세액은  ,  공제되어야 한다. 청구인은  로부터 쟁점토지를  에 양수하기로 하였으며 그   (2)  2018.8.10. OOO OOO ,  에 따라  과 공사대금  부가가치세 별도의 건축계약을 체결하였고 2018.9.1. OOO OOO( ) ,  공사를 진행하여  사용승인을 받았으며 하자 등의 보수가 완료된 후  2
R3. score=0.000 file=국심-2005-전-3976.pdf chunk=0
    [page 1] 문서번호 국심 전 -2005- -3976 결정유형 기각 세목 부가가치세 생산일자 2006. 05. 24. 귀속연도 2003 제목 사실과 다른 세금계산서인지 여부 요지 신축건물 건설 용역의 공급시기를 건축물관리대장상의 건물사용승인일('03.  과 세금계산서 수취일 중 사용승인일로 본 사례 3.17.) ('0
R4. score=0.000 file=국심-2005-전-3976.pdf chunk=1
     및 방수공사등을 계속하여  쟁점세금계산서상의 공급가액이 확정되었으므 2003.5.7.  로 이 건 용역의 공급시기는 부가가치세법시행령 제 조  22 제호의 규정에 따라 정산대금이 확정된 시점인 쟁점세금 3 계산서 수취일 로 보아야 한다 (2003.5.7.) . 설사 이 건 용역의 공급시기를 신축건물의 사용승인  (2) 
R5. score=0.000 file=조심-2025-인-0392.pdf chunk=0
    [page 1] 문서번호 조심 인 -2025- -0392 결정유형 기타 세목 부가가치세 생산일자 2025. 08. 28. 귀속연도 2015 제목 쟁점세금계산서가 사실과 다른 세금계산서인지 여부 요지 처분청은 쟁점매입처의 대표인 윤덕호의 유죄판결 외에는 쟁점거래가 가공 거래라는 입증이 부족해 보이는데 청구인은 거래명세서 장
R6. score=0.000 file=조심-2025-인-0392.pdf chunk=1
    서를 실제 거래 없이 발행 및 수취한 것으로 보아 검찰에 고발하였고,  2016.5.18. 처분청에 과세자료 통보하였다   . 다 처분청은 쟁점세금계산서를 가공세금계산서로 보아   .  , 2024.10.18.  청구인에게  년 제기 부가가치세  원을 경정 고지하였다 2015 2 OOO · . 라청구인은 이에 불복하여  
R7. score=0.000 file=조심-2013-부-2294.pdf chunk=0
    [page 1] 문서번호 조심 부 -2013- -2294 결정유형 기각 세목 부가가치세 생산일자 2013. 07. 11. 귀속연도 2010 제목 쟁점세금계산서를 사실과 다른 세금계산서로 보아 과세한 처분은 정당함 요지 쟁점매입처는 유류저장시설에 유류를 저장한 사실이 없고 출하지 등을 확 ,  인하면 쟁점매입체가 실제사업자
R8. score=0.000 file=조심-2013-부-2294.pdf chunk=1
     (1)  , 2010 리점이 신규 허가등록 되었으며 청구인은  주유소 소장 조 으로부터 쟁점매입 ,  OOO OOO 처의 손 이사를 소개 받아 사업자등록증 및 석유판매등록증 등의 제반 서류를 확 OOO  인하고 거래를 시작하였다 이는 청구인이 정유사로부터 유류를 매입하는 것보다 낮은  .  단가로 매입할 수 있기에 쟁점
R9. score=0.000 file=조심-2012-구-1315.pdf chunk=0
    [page 1] 문서번호 조심 구 -2012- -1315 결정유형 기각 세목 부가가치세 생산일자 2013. 01. 23. 귀속연도 2008 제목 독립된 사업주체와의 거래가 아니므로 매입세액을 불공제함 요지 소사장제의 경우 독립채산제로 손익의 귀속이 명확하게 입증되어야 함에도  이에 대한 입증이 부족하고 소사장이 사업기간 
R10. score=0.000 file=조심-2012-구-1315.pdf chunk=1
    OO,  OOOOO  OOOO  OO,OOO,OOOO,  OOOOO  OOOO  OO,OOO,OOOO,  OOO  OOOOOOOOO  O,OOO,OOOO,  OOOOOOOOO  OOO,OOOO, OOOOOOOOO OO,OOO,OOOO, OOOOOOOOO OO,OOO,OOOO,  을 환급하거나 경정고지하였다 OOOOOOOO
```
</details>


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _24.6s_

사실과 다른 세금계산서 사건들에서 **매입세액 공제가 부인된 핵심 이유**는, 세금계산서의 **형식상 공급자와 실제 공급자가 다르거나**, 아예 **실물거래 없이 세금계산서만 수수된 경우**가 반복적으로 확인되었기 때문입니다. 즉, 부가가치세법상 **필요적 기재사항이 사실과 다른 세금계산서**로 보이면 그에 따른 매입세액은 공제되지 않는다는 구조입니다 [C2][C3][C4].

반복되는 패턴을 보면 다음과 같습니다.

1. **거래처가 자료상으로 확정·고발됨**
   - 거래처가 자료상으로 조사되거나 고발되어 정상적인 사업을 한 흔적이 없고, 실질 공급자로 보기 어려운 경우가 많았습니다 [C4].
   - 이런 경우 명의상 거래처는 실제 공급자가 아니므로, 세금계산서 자체가 사실과 다른 것으로 판단되었습니다 [C4].

2. **실제 공급자와 세금계산서상 공급자가 불일치**
   - 명의상 상대방과 실제 거래 상대방이 다르거나, 실제 공급자가 다른 회사로 조사된 경우가 반복되었습니다 [C1][C3].
   - 대표적으로 공급자 명의는 A회사인데 실제 물품·용역 제공은 B회사로 확인되면, 해당 세금계산서는 사실과 다른 세금계산서가 됩니다 [C1][C3].

3. **실물거래 없이 세금계산서만 수수된 가공거래**
   - 아예 실물거래가 인정되지 않고 세금계산서만 주고받은 경우에는, 그 세금계산서에 기초한 매입세액 공제가 인정되지 않았습니다 [C2].
   - 이 유형에서는 “정상거래” 주장보다 실제 재화·용역의 공급 여부가 핵심이었습니다 [C2].

4. **선의의 거래당사자 인정 실패**
   - 일부 사건에서는 사업자등록증, 법인통장, 출하전표, 계약서 등을 확인했더라도, 실제 공급자가 아님을 알 수 있었거나 주의의무를 다하지 못했다고 보아 공제가 부인되었습니다 [C4].
   - 반대로, 서류 확인·대금 지급 방식·현장 정황 등이 충분해 선량한 관리자의 주의의무를 다했다고 보면 공제가 인정되기도 했습니다 [C1][C5].  
   - 따라서 공제 여부는 단순히 서류가 있었는지보다, **거래상대방의 실체를 확인하려는 주의의무를 다했는지**에 달려 있었습니다 [C1][C5].

증거 유형도 반복됩니다. 자주 등장한 것은 **세금계산서, 사업자등록증, 출하전표, 거래명세표, 계좌이체 내역, 확인서, 통장사본** 등인데, 이 자료들은 주로 거래의 실질과 선의 여부를 다투는 데 쓰였습니다. 다만 서류가 있어도 실제 공급자 불일치나 가공거래가 드러나면 공제는 부인되었습니다 [C2][C4].

**결과 경향**은, 전체 80건 중 **기각이 61건**으로 가장 많아, 사실과 다른 세금계산서 사건에서는 대체로 **매입세액 불공제 처분이 유지되는 방향**이 우세했습니다. 인용은 13건에 그쳤고, 인용된 경우는 대체로 **선의의 거래당사자**와 **주의의무 이행**이 구체적으로 인정된 사례였습니다 [C1][C5].

<details><summary>retrieval log 보기</summary>

```
## Seed Phrase Nodes
1. score=21.63 type=LegalConcept label=사실과 다른 세금계산서
2. score=17.35 type=LegalConcept label=매입세액
3. score=17.35 type=EvidenceType label=매입세액
4. score=16.29 type=FactPattern label=쟁점 세금계산서가 사실과 다름
5. score=16.13 type=LegalConcept label=매입세액 공제
6. score=13.63 type=LegalConcept label=사실과 다른 기재
7. score=13.17 type=FactPattern label=쟁점세금계산서가 사실과 다른 공급자로부터 교부됨
8. score=12.25 type=LegalConcept label=부가가치세법상 사실과 다른 세금계산서

## Graph Retrieval Summary
expanded_phrase_nodes=22
retrieved_issues=80
analysis_mode=overview
accepted_issues=19
rejected_issues=61
other_issues=0

## Representative Cases
Top graph matches:
  [C1] file=국심-2005-중-3430.pdf issue=0 outcome=인용 score=63.32
       쟁점: 청구인이 사실과 다른 세금계산서에 따른 매입세액 공제를 받을 수 있는 선의의 거래자에 해당하는지 여부
       결론: 인용. 청구인은 선의의 거래자로 보아 쟁점세금계산서상의 매입세액을 공제할 수 있다고 결정하였다.
  [C2] file=조심-2015-서-1441.pdf issue=1 outcome=기각 score=54.05
       쟁점: 쟁점 세금계산서의 수수에 따라 부가가치세 매입세액 공제가 가능한지 여부
       결론: 사실과 다른 세금계산서에 기초한 매입세액 공제는 인정되지 않으므로, 환급·공제분을 포함한 부가가치세 경정고지는 적법하다.
  [C3] file=국심-2005-부-0976.pdf issue=0 outcome=기각 score=52.50
       쟁점: 쟁점세금계산서를 공급자가 사실과 다른 세금계산서로 보아 매입세액을 불공제한 처분이 정당한지 여부
       결론: 기각. 쟁점세금계산서는 공급자가 사실과 다른 세금계산서에 해당하여 매입세액 불공제 처분이 정당하다고 보았다.
  [C4] file=국심-2004-중-1157.pdf issue=0 outcome=기각 score=51.85
       쟁점: 쟁점세금계산서의 공급자가 사실과 다른 세금계산서인지, 그리고 청구인이 선의의 거래당사자로서 매입세액 공제를 받을 수 있는지 여부
       결론: 기각, 쟁점세금계산서상의 매입세액 불공제 처분은 정당하다.
  [C5] file=조심-2009-전-3082.pdf issue=0 outcome=인용 score=51.85
       쟁점: 청구인이 쟁점 유류를 거래하면서 선량한 관리자로서의 주의의무를 다한 선의의 거래당사자에 해당하여 사실과 다른 세금계산서의 매입세액 공제를 받을 수 있는지 여부
       결론: 청구인은 선의의 거래당사자에 해당하므로 쟁점매입세금계산서 관련 매입세액 불공제는 잘못이며, 부가가치세 부과처분은 취소되어야 한다.
```
</details>


---


#### 6-4-2. 상담형 질문

In [11]:
results2 = compare("실제 거래는 존재했으나 거래 상대방의 명의만 바뀐 경우, 심판원은 보통 어떤 논리로 과세관청의 처분이 정당하다고 판단해?", show_logs=True)

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Issue-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...


### 질문

> 실제 거래는 존재했으나 거래 상대방의 명의만 바뀐 경우, 심판원은 보통 어떤 논리로 과세관청의 처분이 정당하다고 판단해?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _20.3s_

제공된 검색 결과만 보면, 심판원은 **실제 물품·용역의 흐름이 있었다는 사정만으로는 부족하고**, **세금계산서상 거래상대방이 실제 거래상대방과 일치하는지**를 핵심으로 봅니다.  
즉, **거래 상대방의 명의만 바뀌었는데 실제 대표자·사업자와 달리 기재되었거나, 명의대여자·자료상과의 거래로 확인되면 “사실과 다른 세금계산서”로 보아 과세관청 처분을 정당하다고 판단하는 논리**입니다. [1][3][7][8]

구체적으로는 보통 다음 논리입니다.

1. **실물거래가 있었다고 해도 세금계산서의 상대방이 실제 공급자와 다르면 문제**  
   - 예를 들어 실제 구리 거래가 있었지만, 세금계산서상의 대표자 이름이 사업자등록증상 이름과 달라 재작성된 사안이 보입니다. 이런 경우 심판원은 거래의 실재 여부와 별개로 **세금계산서 기재의 정확성**을 봅니다. [1]

2. **상대방이 자료상·명의대여자라면 처분이 정당**  
   - 유류 거래 사례에서는 쟁점 매입처들이 **실물거래 없이 세금계산서만 발행한 자료상**으로 고발된 업체였고, 심판원은 이런 경우 거래를 부인하거나 가공거래로 볼 수 있다는 흐름을 보입니다. [3][7][8]

3. **납세자가 정상사업자 여부를 충분히 확인하지 못하면 선의 주장도 약해짐**  
   - 검색 결과에서는 사업자등록증 사본, 통장사본 등을 받았더라도, 상대방이 실제로 정상업체인지에 대한 주의의무가 문제되고, 잘못 판단한 경우 납세자 귀책사유가 있다고 봅니다. [1][2][5]

정리하면, **“실제 거래는 있었지만 명의만 바뀌었다”는 사안에서 심판원이 과세관청 편을 드는 전형적 논리는**  
- **실제 공급자와 세금계산서상 공급자가 다르다**,  
- **상대방이 명의대여자·자료상이다**,  
- **납세자가 정상사업자 확인의 주의의무를 다하지 못했다**,  
는 점을 근거로 **세금계산서를 사실과 다른 것으로 본다**는 것입니다. [1][3][7][8]

원하시면 제가 이 답변을 바탕으로 **“심판원 판단요지” 형식으로 3줄 요약**도 해드릴게요.

<details><summary>retrieval log 보기</summary>

```
## Retrieved Chunks
1. score=17.191 file=조심-2011-중-1186.pdf chunk=9
   ○○ ○○○ 사를 하여 자료상확정자로 고발된 업체입니다 당시 조사보고서를 보면 는 사업장 .  , ○○○ 도 없고 당시의 대표자는 명의만 대여한 것으로 되어 있는데 귀하는 이러한 사실을 알았 습니까? 문 전혀 몰랐습니다 급 구리는 고가의 품목이라 조 이 본인에게 매입의사를 타 ( )  . A ○○○ 진할 때도 실제 매입할 구리가 있는지가 중요한 것이었지 어느 업체의 구리인지는 중요 한 것이 
2. score=14.674 file=국심-2000-서-2316.pdf chunk=8
    부가가치세 과세대상이 되는지 여부에 대해서는 어떠한 안내도 받지 못하였던 바,  처분청이 쟁점전화서비스용역을 부가가치세 과세대상이 아니라고 하여 이 건 환급하지  않은 처분은 신의성실의 원칙에 반하는 것이라고 주장한다.  나 일반적으로 조세관계에서 과세관청의 행위에 대하여 신의성실의 원칙이 적용되는 요 ( )  건으로는 첫째과세관청이 납세자에게 신뢰의 대상이 되는 공적인 견해표명을 하여야 
3. score=13.924 file=조심-2013-중-0993.pdf chunk=4
   확인서 유류대금입금거래내역조회서 등을 제출하였다 ,  . 가 청구법인의 주유소는 임대가 아닌 법인의 자산인 까닭에 모든 일에 신중 ( )  하였고 그 결과 거래마다 철저한 확인을 거쳤는바 거래 시작 전 사업자등록증 ,  , 1)  ,  석유판매업등록증 사업자명의의 통장사본 탱크가 없을 경우 탱크임대차계약서를  ,  ,  제출받았고 운전기사가 가져 온 출하전표를 받은 후 전표번호 수송장비번호 
4. score=12.716 file=조심-2025-서-2015.pdf chunk=3
    따라 하도급을 이용하여 시공하는 방법을 선택하였는바 하도급 구조 ,  와 관련된 수많은 사업체를 직접 관리한다면 적지 않은 비용이 소모되므로 하도급 구조에서  ,  용역의 각 구성 부분을 별개 업체가 수행하는 것이 불필요하거나 비합리적이라고 단정할 수  없으며인천지방법원  선고  구합 판결 참조 하도급 공사와 관련하여 실 ( 2023.4.27.  2022 50544  ),  제 대금이 이체
5. score=12.686 file=국심-2000-서-2316.pdf chunk=9
   대상으로 받아들인 것은 그 해석에 문제점이 있어 과세관청의 견해표명이 정당하다고  ,  신뢰한 데에 대하여 납세자에게 귀책사유가 없다고 볼 수 없다.  라 따라서 위의 질의회신문 등을 청구법인이 잘못 판단하여 쟁점전화서비스용역이 부가 ( )  가치세 과세되는 것으로 본 것은 청구법인에게 귀책사유가 있는 것으로 보이고과세당국 ,  의 공적인 의견표명도 없었던 것으로 보아야 하므로,  처분청이 
6. score=11.887 file=조심-2025-서-2015.pdf chunk=4
   누락이나 허위기장도 없었으며 부정한 의도나 선의의 투자자에 대한 피해도 없었 , ⑤  을 뿐만 아니라  관련 조세를 성실히 신고납부하여 어떠한 조세회피 의도나 세금탈루도 없 ⑥  ․ 는 사안으로서 쟁점세금계산서를 사실과 다른 세금계산서로 볼 수 없다 ,  . 마 한편 처분청은 청구법인을 쟁점매입처와 쟁점매출처 사이에 끼워서 쟁점세금계산서를     ( )  발급 수취한 것은 페이퍼컴퍼니인 청구
7. score=11.627 file=조심-2009-서-2362.pdf chunk=2
   2)  ( “ ” ○○ ① ① 다는 실제 거래에 의하여 수취한 정상적인 매입세금계산서인데도 이를 가공거래로  ) 보아 관련 제세를 부과한 처분은 부당하며 청구법인이  로부터 실제 매입하였음 ,  ○○ 을 이의신청 재조사결과에 의해 확인하였음에도 실매입처를  닷컴으로 재경정한  ○○○ 처분은 부당하다.  [page 3] 엔터프라이즈로부터 받은 쟁점 세금계산서이와 관련된 거래를 이하 쟁점 (3)
8. score=11.359 file=조심-2017-서-2188.pdf chunk=0
   [page 1] 문서번호 조심 서 -2017- -2188 결정유형 기각 세목 부가가치세 생산일자 2017. 11. 28. 귀속연도 2017 제목 쟁점 거래를 가공거래로 볼 수 있는지 여부 요지 과세관청의 조사내용에 의하면 청구법인은 실물거래 없이 매출실적을 높이 기 위해 형식상 쟁점 ㆍ 거래를 한 것으로 나타나는 반면 청구법인은 동  ,  ⓛ ② 거래를 실제로 하였다고 주장할 뿐 그에 대한 
9. score=11.287 file=조심-2019-전-0241.pdf chunk=16
   이며그 중에서 건물 매입과 관련한 매입세액을 자진하여 토지 관련  ,  매입세액으로 불공제하여 신고한 것과 동일한 논리로 본다면별도 분리하지 못하여 불가 ,  피하게 매입한 건물에 부속된 쟁점동산도 건물의 일부로 보아 토지 관련 매입세액으로  불공제함이 타당하다. 투자기관이 쟁점부동산 및 쟁점동산의 총 매입비용  원을 지급할 때 토지 매입비     OOO 용의 명목으로 지급한 것으로 보아청구
10. score=11.017 file=조심-2012-중-1198.pdf chunk=9
   고 나타나 있다.  청구법인은 선의의 거래당사자라고 주장하면서 조 의 거래사실확인서 이  (6)  ,  OOO ,  의 사실확인서 박 의 운반사실확인서 백 의 사실확인서 및 주 OOO ,  OOO ,  OOO ( )OOO 정으로부터 청구법인에의 입금내역 및 청구법인에서  자원으로의 송금내역이  OOO 나타난 청구법인의 계좌내역 청구법인 발행의 계량확인서 주 정 자원에  ,  , ( )OOO
```
</details>


---

### ✅ 파이프라인 2 · Issue-based (structural)  · _16.4s_

심판원은 보통 **형식상 명의와 실질이 다르면 실질을 우선** 보되, 그 과정에서 **명의위장 사실을 몰랐고 알지 못한 데 과실도 없었다는 점을 납세자가 입증하지 못하면** 과세관청 처분을 정당하다고 봅니다[1].  

즉, 실제 거래 자체가 있었더라도 거래 상대방의 **명의만 바뀐 경우**에는 단순히 거래가 있었다는 사정만으로는 부족하고,  
- **실제 공급자와 세금계산서상 공급자가 다른지**,  
- **납세자가 그 명의위장을 알았는지/몰랐는지**,  
- **거래 전 확인의무를 다했는지**  
를 따져서, 납세자가 선의의 거래당사자임을 못 증명하면 매입세액 공제나 관련 과세상 이익을 인정하지 않는 논리입니다[1].  

또 실제 사업자 여부가 쟁점인 경우에도, 심판원은 **사업자등록·신고·임차·운영 등 실질적 사정이 청구인에게 있으면 명의대여 주장을 배척**하고 청구인을 실제 사업자로 보아 과세한 사례가 있습니다[4].  

정리하면, **“실제 거래는 있었지만 상대방 명의만 바뀐 경우”에는 실질과세를 이유로, 명의상 당사자가 아니라 실질적 당사자를 기준으로 보되, 납세자가 그 명의위장을 몰랐고 과실이 없었다는 점을 입증하지 못하면 과세관청 처분이 정당하다고 판단하는 경향**입니다[1][4].

<details><summary>retrieval log 보기</summary>

```
## Retrieved Issues
1. score=10.969 file=조심-2015-부-3287.pdf issue=1
   쟁점: 청구인이 쟁점세금계산서를 수취함에 있어 선의의 거래당사자에 해당하는지 여부
   결론: 청구인은 선의의 거래당사자라고 보기 어렵고, 쟁점세금계산서의 매입세액 공제를 배제한 처분은 정당하다.
2. score=9.260 file=조심-2017-구-0609.pdf issue=1
   쟁점: 매입누락액에 부가가치율을 적용하여 매출누락액을 환산한 뒤 부가가치세를 과세한 처분이 근거과세 원칙에 위배되는지 여부
   결론: 기각 — 부가가치율을 적용한 매출누락액 환산 및 부가가치세 과세는 정당하다.
3. score=9.194 file=조심-2010-중-3848.pdf issue=0
   쟁점: 토지, 건물 및 기계장치를 일괄 공급한 경우 토지와 건물 등의 가액이 불분명하다고 보아 장부가액과 감정평가액을 기준으로 안분계산한 처분이 정당한지 여부
   결론: 기각. 토지·건물·기계장치를 일괄 양도한 경우로서 건물 등의 실지거래가액이 불분명하므로, 장부가액과 감정평가액을 기준으로 안분계산한 처분은 정당하다.
4. score=9.048 file=국심-2003-부-1075.pdf issue=0
   쟁점: 쟁점사업장의 사업자를 청구인으로 보아 부가가치세를 과세한 처분이 정당한지, 아니면 청구인의 명의대여 주장대로 실제 사업자를 윤○○으로 보아야 하는지 여부
   결론: 청구인의 명의대여 주장을 받아들이지 않고, 청구인을 쟁점사업장의 실제 사업자로 보아 부가가치세를 과세한 처분이 정당하다고 보아 기각
5. score=8.857 file=국심-2004-중-3885.pdf issue=0
   쟁점: 쟁점세금계산서를 공급시기가 다른 사실과 다른 세금계산서로 보아 매입세액을 불공제한 처분이 정당한지 여부
   결론: 인용. 쟁점세금계산서는 사실과 다른 세금계산서가 아니므로 매입세액 불공제 및 부가가치세 경정고지는 위법하다.

## Added Raw Chunks
R1. score=0.000 file=조심-2015-부-3287.pdf chunk=0
    [page 1] 문서번호 조심 부 -2015- -3287 결정유형 기각 세목 부가가치세 생산일자 2016. 06. 27. 귀속연도 2011 제목 실제 거래 후 쟁점세금계산서를 교부받았으므로 관련 매입세액을 매출세액 에서 공제하여야 한다는 청구주장의 당부 등  요지 붙임과 같습니다.  내용 결정 내용은 붙임과 같습니다. 관
R2. score=0.000 file=조심-2015-부-3287.pdf chunk=1
    산서 수취시  의 사업자등록 여부를  전산망인  를 통해 확인하는 등 쟁점거래 OOO OOO  OOO 주의의무를 다하였으므로 단순히 자료상으로 확정된  과 거래하였다는 이유로 부가가 ,  OOO 세 과세한 이 건 처분은 부당하다. 나처분청 의견   .    (1) 조사청 조사시 청구인의 배우자인  이 쟁점세금계산서를 사실
R3. score=0.000 file=조심-2017-구-0609.pdf chunk=0
    [page 1] 문서번호 조심 구 -2017- -0609 결정유형 기각 세목 부가가치세 생산일자 2017. 04. 10. 귀속연도 2011 제목 매입누락액에 부가가치율을 적용하여 매출누락액을 환산하고 부가가치세    를 과세한 처분의 당부 등  요지 처분청이 사업의 종류별ㆍ지역별로 정한 부가가치율을 적용하여 매출누락   
R4. score=0.000 file=조심-2017-구-0609.pdf chunk=1
    7.1.13.  제기하였다. 청구인 주장 및 처분청 의견 2.  가 청구인 주장 .       처분청은 쟁점사업장의 총 매입대금쟁점계좌로 송금한 금액어음결제액주류 (1)  ( + + 구매카드 결제액에서 쟁점매입처와 주식회사  이하  라 한다로부터 수 ) OOO( “OOO” ) 취한 매입세금계산서 공급대가를 차감하여 매입누
R5. score=0.000 file=조심-2010-중-3848.pdf chunk=0
    [page 1] 문서번호 조심 중 -2010- -3848 결정유형 기각 세목 부가가치세 생산일자 2011. 02. 08. 귀속연도 2010 제목 실지거래가액이 불분명하여 건물가액은 장부가액과 감정평가액 비율로 각  안분하여   산정함 요지 실지거래가액이 불분명하다고 보아 토지와 건물 등을 장부가액과 감정평가 ,  액 비율
R6. score=0.000 file=조심-2010-중-3848.pdf chunk=1
    견 2.  가청구법인 주장   .  청구법인이 취득한 쟁점건물 등은 쟁점매입처가 법인전환전 개인사업자인  대   ( ○○○ 표자  이라 한다이  와 건물신축공사 도급계약을 체결하여 공사를 하면서  ) ○○○ ○○○ 쟁점건물  백만원 쟁점기계장치  백만원을 지급한 사실이 확인되며 가  1,016 ,  66 , ○○○ 공사대
R7. score=0.000 file=국심-2003-부-1075.pdf chunk=0
    [page 1] 문서번호 국심 부 -2003- -1075 결정유형 기각 세목 부가가치세 생산일자 2003. 09. 02. 귀속연도 2002 제목 실제사업자 여부 요지 사업장의 대표로 되어 있으나 명의 대여라는 주장의 사실관계를 판단하는  사례임 내용 결정 내용은 붙임과 같습니다. 관련법령 부가가치세법 제 조 국세기본법 제
R8. score=0.000 file=국심-2003-부-1075.pdf chunk=1
    며 실제 사업자라고 주장하는 윤 은 무재산    ○○○ 결손처분자로서 실제로 쟁점사업장에서 사업을 했는지 여부가 확인되지 않는다. 또한 청구인은 사업자등록과 그동안의 사업내용에 대하여 청구인의 명의로 신고 및    납부를 하였으므로 청구인에게 부가가치세를 결정고지 한 당초 처분은 정당하다. 심리 및 판단 3.  가 쟁점 
R9. score=0.000 file=국심-2004-중-3885.pdf chunk=0
    [page 1] 문서번호 국심 중 -2004- -3885 결정유형 인용 세목 부가가치세 생산일자 2005. 01. 19. 귀속연도 2004 제목 중간지급 조건부 거래 해당 여부 요지 당초계약서 내용은 계약일부터 잔금지급일까지의 기간이 월 미만이었으나 6 ,  준공 전에 추가 변경계약을 체결하여 그 기간이 월 이상이 된 경
R10. score=0.000 file=국심-2004-중-3885.pdf chunk=1
    전등기하였으므로  교부받은 쟁점세금 2004.4.30.  계산서는 정당한 세금계산서에 해당함에도 이를 사실과  다른 세금계산서로 보아 매입세액을 불공제한 처분은 부 당하다. 나 처분청 의견   .  추가 계약내용은 청구인이 당초 계약내용대로 잔금을 지   급하지 않음으로써 미납잔금에 대한 지급기한을 연장한  것으로 당초
```
</details>


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _20.4s_

제공된 그래프 패턴만 보면, 심판원은 **“실제 거래가 있었다”는 주장보다 거래의 실질과 객관적 증빙이 맞는지**를 먼저 봅니다.  
특히 **거래상대방의 명의만 바뀌었는데 세금계산서·거래명세서·출하전표·금융흐름이 그 변경을 따라가지 못하면**, 과세관청 처분을 대체로 정당하다고 판단하는 경향이 뚜렷합니다.

핵심 논리는 다음과 같이 반복됩니다.

1. **명의 변경만 있고 실질 거래의 흐름이 불명확함**
   - 법인등기부상 상호·소재지가 바뀌었는데도, 청구인이 제출한 세금계산서와 거래명세서에 **변경 전 상호·소재지**가 기재되어 있으면, 실제 거래의 상대방과 문서상 상대방이 일치하지 않는다고 봅니다.  
   - 이런 경우 심판원은 거래의 실질을 의심합니다. [C4]

2. **객관적 증빙 부족**
   - 단순히 “실제 거래였다”는 주장만 있고, **금융증빙·반품증빙·입출고 자료** 같은 구체적 자료가 없으면 실제 거래로 인정하지 않습니다.  
   - 즉, 명의 변경 주장이 있어도 입증 책임은 납세자에게 있고, 이를 못 대면 가공거래 또는 사실과 다른 세금계산서로 봅니다. [C1] [C2]

3. **상대방이 자료상으로 고발된 사정**
   - 거래 상대방이 **자료상으로 고발된 사업자**이면, 심판원은 그 자체를 강한 의심 사유로 보고 추가 입증을 요구합니다.  
   - 실거래를 입증할 만한 객관적 자료가 없으면 과세관청 처분을 유지합니다. [C1] [C2] [C4]

4. **서류의 외형보다 실질 흐름 중시**
   - 출하전표 기재가 부실하거나, 저장시설에서 실제 입출고가 확인되지 않거나, 자금흐름이 거래대금으로 연결되지 않으면, 형식상 명의만 바뀐 것으로 보지 않고 **실질 거래 자체를 부정**합니다. [C4]

5. **결론 경향: 기각이 다수**
   - 전체 17건 중 **기각 10건**으로, 실제 거래나 명의 변경을 주장해도 **객관적 증빙이 부족하면 처분이 정당**하다고 보는 흐름이 우세합니다.  
   - 반대로 인용된 경우는, 별도 쟁점에서 실제 거래로 인정되거나(예: 쟁점광고비 실제 거래 판단) 주위적 쟁점이 받아들여져 예비적 쟁점을 따로 볼 필요가 없었던 사례입니다. [C3] [C5]

정리하면, 심판원은 보통 **“명의 변경은 형식일 뿐이고, 실제 공급·인수·대금지급의 실질이 입증되지 않으면 처분은 정당하다”**는 논리로 판단합니다.  
즉, **실제 거래 주장 + 명의 변경 주장만으로는 부족하고, 문서·금융·물류 증빙이 서로 맞아떨어져야** 납세자 주장이 받아들여집니다.

<details><summary>retrieval log 보기</summary>

```
## Seed Phrase Nodes
1. score=18.71 type=LegalConcept label=실제 거래
2. score=13.72 type=LegalConcept label=명의만 있는 경우 사실상 귀속자
3. score=11.38 type=EvidenceType label=과세관청의 유사 거래 처분 사례
4. score=10.93 type=FactPattern label=환급신청을 정당하다고 처리함
5. score=10.32 type=FactPattern label=다른 거래는 전자세금계산서로 발급
6. score=9.93 type=EvidenceType label=과세관청의 미지적
7. score=9.54 type=FactPattern label=거래 전체를 허위로 본 경우
8. score=9.33 type=FactPattern label=실제 거래 주장

## Graph Retrieval Summary
expanded_phrase_nodes=14
retrieved_issues=17
analysis_mode=overview
accepted_issues=4
rejected_issues=10
other_issues=3

## Representative Cases
Top graph matches:
  [C1] file=조심-2008-서-0568.pdf issue=0 outcome=기각 score=26.23
       쟁점: 쟁점세금계산서1 관련 거래를 실제거래로 보아 부가가치세 과세표준에 가산할 수 있는지 여부
       결론: 실제거래로 인정되지 않아 가공세금계산서로 보았고, 처분은 정당하다.
  [C2] file=조심-2008-서-0568.pdf issue=1 outcome=기각 score=25.09
       쟁점: 쟁점세금계산서2 및 쟁점세금계산서3 관련 거래를 실제거래로 보아 매입세액 공제가 가능한지 여부
       결론: 실제거래로 인정되지 않아 관련 매입세액은 불공제 대상이 되었다.
  [C3] file=국심-2006-중-0446.pdf issue=2 outcome=중복 상여처분 주장 배척 score=18.71
       쟁점: 쟁점광고비와 관련하여 청구법인 계좌 인출액을 근거로 한 대표자 등 상여처분이 쟁점광고비에 대한 상여처분과 중복되는지 여부
       결론: 중복 상여처분 주장 배척
  [C4] file=조심-2009-전-3254.pdf issue=0 outcome=기각 score=18.71
       쟁점: 쟁점세금계산서가 실제 거래에 따라 교부받은 것인지, 또는 실제 거래가 아니더라도 청구인이 선의의 거래당사자로서 매입세액 공제를 받을 수 있는지 여부
       결론: 실제 거래로 보기 어렵고 선의의 거래당사자에도 해당하지 않으므로 쟁점세금계산서 관련 매입세액 불공제 처분은 정당하며, 심판청구는 기각
  [C5] file=조심-2011-서-1120.pdf issue=1 outcome=실익 없음 score=18.71
       쟁점: 익명거래의 대리·위임에 따른 정상적인 세금계산서 발행으로 볼 수 있는지 여부
       결론: 실익 없음. 주위적 청구 인용으로 별도 판단하지 않았다.
```
</details>


---


#### 6-4-3. 패턴형 질문

In [12]:
results3 = compare("세무조사 나와서 억울하게 엮인 부가가치세 심판 사건들 보면, 보통 법원에서 어떤 증거를 가장 중요하게 봐줘?", show_logs=True)

▶ 파이프라인 1 · Chunk BM25 (baseline) 실행 중...
▶ 파이프라인 2 · Issue-based (structural) 실행 중...
▶ 파이프라인 3 · Graph RAG (relational) 실행 중...


### 질문

> 세무조사 나와서 억울하게 엮인 부가가치세 심판 사건들 보면, 보통 법원에서 어떤 증거를 가장 중요하게 봐줘?

### ✅ 파이프라인 1 · Chunk BM25 (baseline)  · _27.1s_

제공된 검색 결과만 보면, 부가가치세 심판 사건에서 법원이 가장 중요하게 보는 건 **“실제 거래가 있었는지”를 뒷받침하는 객관적 증빙**입니다. 특히 아래 자료들이 핵심으로 보입니다.

1. **정산서**
   - 거래대금이나 수수료를 최종적으로 확정하는 **가장 중요한 최종 증빙**으로 언급됩니다. 정산서가 세금계산서 수수내역, 금융증빙과 맞아떨어지면 실제 거래를 뒷받침하는 자료가 됩니다. [1]

2. **금융증빙(계좌이체 내역, 입출금 내역 등)**
   - 실제 돈이 오갔는지 확인할 수 있는 자료가 중요합니다. 다만, 금융거래 내역이 있다고 해서 그것만으로 곧바로 용역 실체가 인정되는 것은 아니고, 거래 구조 전체와 함께 봅니다. [1][8]

3. **세금계산서와 그 내용의 일치 여부**
   - 세금계산서가 정산서, 거래내역, 자금흐름과 서로 맞는지 중요하게 봅니다. 반대로, 객관적 대금지급증빙이 없으면 사실과 다른 세금계산서로 판단될 수 있습니다. [1][4]

4. **구체적인 거래자료**
   - 매입누락이나 가공거래가 쟁점인 사건에서는 **거래명세서, 상품 종류·수량·단가, 구매내역** 같은 구체적 자료가 중요하다고 보입니다. 단순히 계좌거래나 금전흐름만으로는 부족하다고 다투는 사례도 있습니다. [2]

정리하면, 이 결과들만 기준으로는 법원이 **진술보다 객관적 서류**, 그중에서도 **정산서 + 금융증빙 + 세금계산서의 일치**를 가장 중시한다고 볼 수 있습니다.  
반대로, **계약서나 주장만 있고 실제 지급·거래를 입증할 자료가 부족하면** 받아들여지기 어렵습니다. [4][8]

원하시면 제가 이걸 바탕으로 **“세무조사에서 억울하게 엮였을 때 준비해야 할 증거 체크리스트”** 형태로도 정리해드릴게요.

<details><summary>retrieval log 보기</summary>

```
## Retrieved Chunks
1. score=12.379 file=조심-2025-구-0998.pdf chunk=3
   하여 정산하고 세금계산서를 수취하고 다 결국 당사자간 수수료를 확정함에 있어 정산서는 가장 중요한 최종 증빙으로서 해당 정산서는 세금계 .  서 수수내역와 일치하며관련 금융증빙을 통하여 실제 입출금내역을 확인할 수 있다 ,  . 나    ( ) 국내 면세점에서 상품을 구매하는 따이공은 어느 특정여행사에 계속하여 종속되는 가이드나 직 신분이 아니다 이들 따이공은 가이드와의 친분이나 구매대행업을
2. score=12.015 file=조심-2017-구-0609.pdf chunk=1
   7.1.13.  제기하였다. 청구인 주장 및 처분청 의견 2.  가 청구인 주장 .       처분청은 쟁점사업장의 총 매입대금쟁점계좌로 송금한 금액어음결제액주류 (1)  ( + + 구매카드 결제액에서 쟁점매입처와 주식회사  이하  라 한다로부터 수 ) OOO( “OOO” ) 취한 매입세금계산서 공급대가를 차감하여 매입누락액을 산정하였는바 년 제 , 2011 기부터  년 제기분 매입누락액을 
3. score=11.363 file=조심-2019-구-4113.pdf chunk=6
    같다에게 조사를 시작하기  일 전에 조사 .  ) 15 상 세목조사기간 및 조사 사유 그 밖에 대통령령으로 정하는 사항을 통지이하 이 조에 ,  ,  ( 사전통지라 한다하여야 한다 다만사전통지를 하면 증거인멸 등으로 조사 목적을 달성 " " ) .  ,  수 없다고 인정되는 경우에는 그러하지 아니하다.  [page 8] 제 조의 세무조사 기간 세무공무원은 조사대상 세목ㆍ업종ㆍ규모조사 난이도
4. score=9.487 file=국심-2003-서-3244.pdf chunk=3
      ,  소명시 가공매입으로 소명하였다가 심판청구시에는 추가 항변자료를 제출하여 입증자료의 부족으로 억울하게 가공 매입으로 부과되었다고 주장만 할 뿐 물건을 실지 매입한  무통장입금증 등의 객관적인 대금지급증빙도 전혀 제시를  못하고 있어 청구인의 주장을 받아들일 수 없다. 따라서 쟁점매입세금계산서를 사실과 다른 세금계산서   ,  [page 5] 로 보아 관련 매입세액을 불공제하고 일부 
5. score=9.196 file=조심-2020-서-8044.pdf chunk=4
   age 4] 과 이 사건 물품 판매계약을 체결함으로써 국내 제강사들에게 직접 이 사건 물품을 공급할 의 무를 부담하였고 이를 토대로 홍콩법인과 이 사건 물품의 수입계약을 체결하였으며 실제 이 ,  ,  행과정에서도 국내 자회사는 국내 제강사들에 대한 이 사건 물품의 공급자로서 계약상 책임을  이행하였다는 점 등을 들어 합금철 수출입거래의 당사자는 실제로 계약을 체결한 홍콩법인과  국내 자회사
6. score=9.108 file=조심-2008-전-1943.pdf chunk=10
   와 동일한 바 위 주문 및 그결정 이유로 볼 때 위 심판결정에서는 청구인이 주장 ,  하는 공동구매사실을 인정한 면이 없지는 않으나 공동구매액의 규모를 구체적으로 적 시하여 재조사 결정하도록 한 것이 아닌 점으로 보면위 심판결정이 공동구매사실을  ,  인정한 다음 그 금액만을 조사확정하라고 내린 결정으로 보기 어려운 면이 있고 또 ,  ․ 한  지방국세청장의 재조사 결정이 비록 청구인의 주장
7. score=8.768 file=조심-2016-중-3611.pdf chunk=4
    년 제기 부가가치세 부    ( )  2012 2 과처분에 대하여 취소소송을 제기하였고 지방법원은  체납 , OOO OOO  법인의 청구를 기각하는 판결을 선고한 것으로 나타나며 청구인의  ,  세무조사 당시 진술과 관련된 판결서 내용은 내용은 아래와 같다. 마 청구인은  이 세무조사 등을 대비하여 청구인에게 체납    ( )  OOO 법인 현황계약내용 등을 숙지시킨 사실이 있고청구인 등과
8. score=8.457 file=조심-2025-구-0998.pdf chunk=6
   가치세를 무신고하거나 신고 후 고액 체납 등을 발생시키고 업하는 폭탄업체로 변질되었다.   (1) 청구법인이 제출한 계약서와 정산서만으로는 제공받았거나 제공한 용역의 실체를 확인할 수 없다. 가 청구법인은 상하위여행사와의 거래에 있어 사전에 합의한 수수료율을 적용하여 정산된 금액을    ( )  · 금계산서 발행금액으로 확정하고 있고 이를 확정하는 과정에서 상호 간 합의를 위해 정산서를 별도
9. score=8.354 file=조심-2025-구-0998.pdf chunk=7
   가공거래로 확정되었다청구법인은 실제 용역 ,  .  제공하거나 제공받지 않고 쟁점세금계산서만을 수수하는 도관으로써의 역할만을 수행하였다고 볼 수 있  [page 6] 며 동일 쟁점에 대한 다수의 심판 결정례에서도 이와 같은 거래형태를 실물거래 없는 가공거래로 판단하 ,  다 이 건과 쟁점이 동일한 다수의 심판 결정례에 따르면 매입세액공제를 목적으로 따이공 몫의 판매장 .  “ 금 지급을 위하
10. score=8.314 file=조심-2009-서-1673.pdf chunk=6
   는 관련없는 기존의 환 ,  급금청구를 이유로 이 건 경정청구를 거부함은 부당하다. 쟁점판결은 청구법인의 범칙금대납업이 기초사회질서에 반하는 유사수신행위로 (4)  서 효력규정을 위반하여 그 법률행위 자체가 무효라고 판결하였고 무효인 법률행 ,  위에 기초한 부가가치세 과세대상 거래행위는 당연무효에 해당하고 이에 따라 당 ,  초의 신고납부는 적법한 법률행위를 전제로 한 것이므로 당연히 경정
```
</details>


---

### ✅ 파이프라인 2 · Issue-based (structural)  · _13.7s_

제공된 사례들만 보면, 부가가치세 심판에서 가장 중요하게 보는 증거는 **“실제 거래가 있었다는 객관적 증빙”**입니다. 단순 주장이나 내부 정리자료만으로는 부족하고, **대금지급증빙, 거래명세, 재고수불, 거래처원장, 매출·매입 내역**처럼 서로 맞물리는 자료가 있어야 설득력이 큽니다 [1][3][4].

특히 다음 자료들이 중요하게 다뤄집니다.

1. **실물의 실제 이동을 보여주는 자료**
   - 실제 납품·매입 사실을 뒷받침하는 증거
   - 예: 매입품목·수량내역, 재고수불현황, 매출상세내역 [1]

2. **객관적인 대금 지급·수수 증빙**
   - 계좌이체 내역, 송금기록 등
   - 다만 이것만으로는 부족할 수 있고, 실제 물품 거래와 함께 봅니다 [1][3][4]

3. **장부 및 거래 흐름이 일관되는 자료**
   - 거래처원장, 거래명세서, 매입·매출 연결자료
   - 실제 사용·소비·판매 흐름이 맞아떨어져야 합니다 [1][3][4]

4. **세무조사에서의 반대사실**
   - 상대방 거래처가 자료상으로 확정되었다거나, 다른 사건에서도 가공거래로 판단된 경우는 청구인에게 불리하게 작용했습니다 [1]

요약하면, 이들 사례에서는 **“객관적이고 외부적으로 확인 가능한 실거래 증빙”**이 핵심이고, 그중에서도 **대금지급 증빙 + 실물거래 증빙 + 장부상 일치 여부**가 함께 갖춰져야 인정될 가능성이 높습니다 [1][3][4][5].

원하시면 제가 이걸 바탕으로  
**“세무조사에서 억울하게 걸렸을 때 준비해야 할 증거 체크리스트”** 형태로 정리해드릴게요.

<details><summary>retrieval log 보기</summary>

```
## Retrieved Issues
1. score=8.727 file=조심-2017-서-0357.pdf issue=0
   쟁점: 쟁점거래처로부터 쟁점세금계산서상 공급가액 상당의 실물 거래가 있었는지 여부
   결론: 청구주장을 받아들이지 아니하고, 쟁점거래처와 쟁점세금계산서상 공급가액 상당의 실제 거래가 없다고 보아 매입세액 불공제 처분을 유지하였다.
2. score=8.129 file=조심-2019-전-3818.pdf issue=2
   쟁점: 세무조사 후 직권으로 감액경정된 세액에 대해 제기한 심판청구가 적법한지 여부
   결론: 각하. 직권 감액경정으로 불복대상이 없어 심판청구의 이 부분은 부적법하다고 판단하였다.
3. score=7.787 file=조심-2017-구-0609.pdf issue=1
   쟁점: 매입누락액에 부가가치율을 적용하여 매출누락액을 환산한 뒤 부가가치세를 과세한 처분이 근거과세 원칙에 위배되는지 여부
   결론: 기각 — 부가가치율을 적용한 매출누락액 환산 및 부가가치세 과세는 정당하다.
4. score=7.042 file=국심-2003-서-3244.pdf issue=1
   쟁점: 가공매입으로 본 금액을 손금불산입하여 법인세를 과세한 처분이 정당한지 여부
   결론: 기각. 일부 가공매입분을 손금불산입하여 법인세를 과세한 처분이 정당하다고 결정하였다.
5. score=5.657 file=조심-2017-서-2492.pdf issue=0
   쟁점: 매출에누리에 해당하는 구매적립 포인트 사용액을 별도로 이력관리하지 아니하고 월별 적립비율에 따라 산정한 금액으로 경정청구한 것이 적법한지 여부
   결론: 경정청구 거부처분은 정당하며 청구를 기각

## Added Raw Chunks
R1. score=0.000 file=조심-2017-서-0357.pdf chunk=0
    [page 1] 문서번호 조심 서 -2017- -0357 결정유형 기각 세목 부가가치세 생산일자 2017. 04. 18. 귀속연도 2017 제목 쟁점거래처와 쟁점세금계산서상 공급가액 상당의 실제 거래가 있었는지 요지 제출된 증빙만으로는 쟁점세금계산서와 관련한 실지거래가 있었다고 단정하 기 어려워 청구주장을 받아들이기 어려
R2. score=0.000 file=조심-2017-서-0357.pdf chunk=1
    의견    .  쟁점거래처에 대한 처분청의 세무조사 결과 쟁점거래처는  년 제기 과세기간     ,  2013 1 동안 실물거래 없이 세금계산서만 유통시킨 자료상 으로 확정된 바 있고 청구법인이  ,  쟁점거래처로부터 쟁점세금계산서상 공급가액 상당의 실물을 실제 매입하였다는 객관 적인 증빙을 제시하지 못하고 있으며 가공거
R3. score=0.000 file=조심-2019-전-3818.pdf chunk=0
    [page 1] 문서번호 조심 전 -2019- -3818 결정유형 재조사 세목 부가가치세 생산일자 2020. 03. 23. 귀속연도 2014 제목 사기나 그 밖의 부정행위에 해당하는지 여부 등 요지 쟁점계좌는 청구인의 배우자 계좌로서 금융조사를 통해 쉽게 발견될 수 있 고청구인들이 쟁점계좌를 통해 수수한 거래대금에 대한 
R4. score=0.000 file=조심-2019-전-3818.pdf chunk=1
     중  년 인출액을 제외한     4. < 3>  OOO 2013 나머지 금액이 사업과 관련하여 지출된 것인지를 청구인들이 제출한 자료를 토대 로 재조사하여 해당 과세연도의 과세표준 세액 및 소득금액변동통지금액을 경정 ,  하며,  나머지 심판청구는 기각한다    5.  . 이 유 처분개요 1.  가 청구인  이하 청구인
R5. score=0.000 file=조심-2017-구-0609.pdf chunk=0
    [page 1] 문서번호 조심 구 -2017- -0609 결정유형 기각 세목 부가가치세 생산일자 2017. 04. 10. 귀속연도 2011 제목 매입누락액에 부가가치율을 적용하여 매출누락액을 환산하고 부가가치세    를 과세한 처분의 당부 등  요지 처분청이 사업의 종류별ㆍ지역별로 정한 부가가치율을 적용하여 매출누락   
R6. score=0.000 file=조심-2017-구-0609.pdf chunk=1
    7.1.13.  제기하였다. 청구인 주장 및 처분청 의견 2.  가 청구인 주장 .       처분청은 쟁점사업장의 총 매입대금쟁점계좌로 송금한 금액어음결제액주류 (1)  ( + + 구매카드 결제액에서 쟁점매입처와 주식회사  이하  라 한다로부터 수 ) OOO( “OOO” ) 취한 매입세금계산서 공급대가를 차감하여 매입누
R7. score=0.000 file=국심-2003-서-3244.pdf chunk=0
    [page 1] 문서번호 국심 서 -2003- -3244 결정유형 기각 세목 부가가치세 생산일자 2004. 02. 06. 귀속연도 2001 제목 가공세금계산서로 보아 매입세액불공제하고 법인세과세한 처분의 당부 요지 이 건의 경우 청구법인은 사업자로서 선량한 관리자로서의 주의의무를 다하 였다고 보기 어렵고 청구법인이 물건을
R8. score=0.000 file=국심-2003-서-3244.pdf chunk=1
    공세금계산서라 하더라도 실지    거래한 공급업체가 교부한 세금계산서이므로 청구법인이  아닌 실지거래처인  에게 부가가치세를 과세하여 ㈜○○○ 야 한다고 주장하나 청구법인은 쟁점매입세금계산서 명 ,  의자인  와 실지거래처인  가 다르다는 사 ㈜○○○ ㈜○○○ 실을 인지하고 위장 또는 가공매입세금계산서를 수취하여  부가가
R9. score=0.000 file=조심-2017-서-2492.pdf chunk=0
    [page 1] 문서번호 조심 서 -2017- -2492 결정유형 기각 세목 부가가치세 생산일자 2017. 12. 20. 귀속연도 2017 제목 이 건 심판청구가 적법한지 여부 요지 청구법인이 산정한 금액은 실제 사용액이 아닌 추정금액에 불과한 점 청구 ,  법인 내부의 문제에 불과한 점실제 포인트 사용 비율을 왜곡할 수
R10. score=0.000 file=조심-2017-서-2492.pdf chunk=1
    하였다   .  2017.4.25.  . 청구법인 주장 및 처분청 의견 2.  가 청구법인 주장   .  처분청은 청구법인이 정한 각 포인트 사용액 산출 방법으로는 고객이 실제 사용한     포인트가 어느 포인트인지 확인할 수 없다는 이유로 경정청구를 거부하였으나 청구법 ,  인과 같은 전자상거래인 온라인몰의 특성상 고객
```
</details>


---

### ✅ 파이프라인 3 · Graph RAG (relational)  · _22.4s_

세무조사로 시작된 부가가치세 사건들에서는, 법원이 **“조사 자체의 적법성”보다도 실제 과세사실을 뒷받침하는 객관적 자료가 있는지**를 가장 중요하게 보는 경향이 뚜렷합니다. 특히 반복되는 패턴은 다음과 같습니다.

1. **세무조사 과정의 적법성은 절차상 자료로 판단**
   - 조사통지서, 납세자권리헌장 교부, 조사사유 설명, 임의제출·예치 여부 같은 **세무조사 절차 관련 자료**가 있으면 절차 위법 주장은 잘 받아들여지지 않는 편입니다.  
   - 예: 관리책임자에게 통지·설명했고 서류도 임의제출 및 예치 형태였다고 보아 절차 하자를 부정함 [C1]

2. **재조사 금지 주장에는 “별도의 조사였는지”를 입증할 객관적 증빙이 핵심**
   - 단순한 전화 문의, 대표 면담, 내부 검토, 추가 자료 요구만으로는 곧바로 재조사로 보지 않고, **별도 세무조사가 실제로 있었는지에 대한 객관적 증빙**을 중요하게 봅니다.  
   - 예: 후속 행위가 있었더라도 내부 검토 수준에 그쳤다고 보아 재조사 위반을 인정하지 않음 [C2]

3. **실질 과세 여부에서는 세금계산서만이 아니라 거래의 흐름을 뒷받침하는 주변 객관증거가 결정적**
   - 세금계산서 자체보다도 **이메일, 거래명세표, 퀵서비스 인수기록, 계좌이체 내역, 장부, 통장, 계약서, 사업자등록 관련 자료**처럼 거래의 실재를 연결하는 자료를 중시합니다.  
   - 반대로 처분청이 “실물거래 없는 가공거래”라고 주장하더라도, 이를 뒤집을 정도의 객관적 증거가 있으면 과세가 취소될 수 있습니다.  
   - 예: 주문 내용·단가·수량이 적힌 이메일, 납품일과 인수일 일치, 대금 흐름, 무혐의·불기소 자료 등을 종합해 실물거래를 인정하고 과세를 취소 [C3]

4. **세무서 쪽이 매출누락·매입누락을 추정할 때는 반대측의 반증자료 유무가 중요**
   - 계좌거래, 조사자료, 거래처 조사 결과 등을 토대로 매입누락이나 매출누락을 추정한 뒤 부가가치율을 적용하는 경우, 납세자가 이를 뒤집으려면 **거래명세서, 거래처원장, 재고수량 확인자료, 자금대여·상환을 입증하는 객관적 증빙**이 필요합니다.  
   - 이런 자료가 없으면 추정과 환산이 받아들여지는 경향이 강합니다.  
   - 예: 계좌이체 및 조사자료를 근거로 한 매입누락 인정, 납세자의 반증 부재, 부가가치율 적용 환산을 정당하다고 판단 [C4]

정리하면, 이 그래프에서 반복되는 핵심은 **“주장”보다 “객관적 증빙”**입니다.  
법원이 특히 무게를 두는 증거는 다음 순서로 보입니다.

- **거래 실재를 직접 보여주는 자료**: 이메일, 거래명세표, 계약서, 세금계산서, 계좌이체 내역, 퀵서비스 인수기록 [C3], [C4]
- **세무조사 적법성을 뒷받침하는 자료**: 조사통지서, 권리헌장, 조사사유 설명, 임의제출·예치 관련 자료 [C1]
- **반증이 없음을 보여주는 자료**: 거래명세서, 원장, 재고자료, 자금흐름 입증자료 부재 [C4]
- **외부 사건 자료**: 고소·불기소·무혐의 처분 자료는 단독으로 결정적이진 않지만, 가공거래 주장에 대한 보조증거로 활용될 수 있음 [C3]

전체 outcome 경향도 이를 반영합니다. 제시된 80건 중 **기각이 63건으로 압도적으로 많고**, 인용은 11건에 그칩니다. 즉, 세무조사에서 과세가 나온 사건은 대체로 **처분청의 조사자료와 계좌·장부 등 객관증거가 있으면 유지**되고, **그에 맞서는 쪽이 거래 실재나 절차 위법을 객관적으로 입증할 때만 뒤집히는 구조**가 반복됩니다.

<details><summary>retrieval log 보기</summary>

```
## Seed Phrase Nodes
1. score=18.59 type=EvidenceType label=세무조사
2. score=18.59 type=LegalConcept label=세무조사
3. score=15.44 type=LegalConcept label=부가가치세
4. score=9.88 type=FactPattern label=주금납입 가장
5. score=9.43 type=EvidenceType label=심판 과정 진술
6. score=9.17 type=EvidenceType label=세무조사 결과
7. score=9.17 type=EvidenceType label=세무조사 복명서
8. score=9.17 type=EvidenceType label=세무조사 자료

## Graph Retrieval Summary
expanded_phrase_nodes=21
retrieved_issues=80
analysis_mode=overview
accepted_issues=15
rejected_issues=63
other_issues=2

## Representative Cases
Top graph matches:
  [C1] file=국심-2000-중-2225.pdf issue=0 outcome=기각 score=33.14
       쟁점: 사업자의 부재 중 사전 동의 없이 조사에 착수하고 서류를 예치한 조사절차상 하자가 있어 부과처분이 무효인지 여부
       결론: 부과처분은 정당하며, 조사절차상 하자 주장은 받아들여지지 않음.
  [C2] file=조심-2019-구-4113.pdf issue=0 outcome=기각 score=32.78
       쟁점: 쟁점세무조사 종료 후 처분청이 쟁점매출누락금액을 추가로 매출누락으로 확정한 것이 재조사 금지 원칙에 위반되는지 여부
       결론: 기각. 쟁점세무조사 종료 후의 행위가 재조사 금지에 위반된다고 보기 어렵다.
  [C3] file=조심-2014-서-4445.pdf issue=0 outcome=인용 score=26.79
       쟁점: 쟁점세금계산서를 실물거래 없는 사실과 다른 세금계산서로 보아 부가가치세를 과세한 처분이 정당한지 여부
       결론: 청구주장을 받아들여, 쟁점세금계산서를 실물거래 없는 사실과 다른 세금계산서로 보아 과세한 부가가치세 부과처분을 취소한 인용 결정이다.
  [C4] file=조심-2017-구-0609.pdf issue=1 outcome=기각 score=26.79
       쟁점: 매입누락액에 부가가치율을 적용하여 매출누락액을 환산한 뒤 부가가치세를 과세한 처분이 근거과세 원칙에 위배되는지 여부
       결론: 기각 — 부가가치율을 적용한 매출누락액 환산 및 부가가치세 과세는 정당하다.
  [C5] file=국심-2000-서-2316.pdf issue=0 outcome=기각 score=26.73
       쟁점: 청구법인이 고속통신망서비스(ISDN, ADSL)와 함께 제공한 음성전화서비스용역이 부가가치세 과세대상인지 여부
       결론: 기각. 음성전화서비스용역은 부가가치세 면제되는 가입전화용역이므로 과세대상이 아니다.
```
</details>


---


### 6-5. 성능지표 비교

In [13]:
metrics1 = compute_metrics(results1, show_log_excerpt=True)

-- P1 retrieval log 일부 (파싱된 10건) --
## Retrieved Chunks
1. score=16.712 file=조심-2019-전-0283.pdf chunk=4
   법인이 쟁점세금계산서를 뒤늦게 교부 받은 것에 아무런 귀책사유가 없으며 또한 ,  기에는 세금을 탈루하겠다는 등의 어떠한 불법적인 의도도 없었다.  라 따라서 대법원 판결의 별개의견과 같이 세금계산서의 기재사항에 따라 거래사    ( )  ,  이 확인되고 부가가치세의 거래징수도 정상적으로 이루어졌으나 납세의무자의 탓으로  리기 어려운 특별한 사정으로 인하여 그 거래시기가 속하는 과세기간 내
2. score=14.147 file=조심-2019-전-0283.pdf chunk=2
   거래에 대한 매출세액이 누락된 사실     3) OOO 발견한 후 수정신고를 통하여 이를 바로잡고자 하

-- P2 retrieval log 일부 (파싱된 15건) --
## Retrieved Issues
1. score=14.912 file=조심-2020-중-2393.pdf issue=0
   쟁점: 쟁점세금계산서가 실제 공급자와 세금계산서상 공급자가 다른 사실과 다른 세금계산서에 해당하여 매입세액 공제가 배제되는지 여부
   결론: 쟁점세금계산서는 사실과 다른 세금계산서에 해당하므로 관련 매입세액은 공제되지 않으며, 경정청구 거부처분은 정당하다고 보아 청구를 기각하였다.
2. score=11.360 file=국심-2005-전-3976.pdf issue=1
   쟁점: 건물 사용승인일과 세금계산서 수취일이 같은 과세기간에 속하여 매입세액 공제가 가능한지 여부
   결론: 기각. 사용승인일과 세금계산서 수취일은 동일한 과세기간에 속하지 않으므로 매입세액 공제 대상이 아니다.

-- P3 retrieval log 일부 (파싱된 5건) --
## Seed Phrase Nodes
1. score=21.63 type=LegalConcept label=사실과 다른 세금계산서
2. score=17.35 t

**Score 분포 / 확신도**

| 파이프라인 | top-1 score | top-5 평균 | margin | 검색 건수 | latency(s) |
|---|---|---|---|---|---|
| P1 | 16.712 | 14.008 | 2.704 | 10 | 17.7 |
| P2 | 14.912 | 11.798 | 3.114 | 15 | 17.8 |
| P3 | 63.32 | 54.714 | 8.606 | 5 | 24.6 |

**사건 겹침도 (Jaccard)**

| 비교 | 겹친 사건 수 | Jaccard |
|---|---|---|
| P1 ∩ P2 | 2 | 0.18 |
| P1 ∩ P3 | 0 | 0.00 |
| P2 ∩ P3 | 0 | 0.00 |

In [14]:
metrics2 = compute_metrics(results2, show_log_excerpt=True)

-- P1 retrieval log 일부 (파싱된 10건) --
## Retrieved Chunks
1. score=17.191 file=조심-2011-중-1186.pdf chunk=9
   ○○ ○○○ 사를 하여 자료상확정자로 고발된 업체입니다 당시 조사보고서를 보면 는 사업장 .  , ○○○ 도 없고 당시의 대표자는 명의만 대여한 것으로 되어 있는데 귀하는 이러한 사실을 알았 습니까? 문 전혀 몰랐습니다 급 구리는 고가의 품목이라 조 이 본인에게 매입의사를 타 ( )  . A ○○○ 진할 때도 실제 매입할 구리가 있는지가 중요한 것이었지 어느 업체의 구리인지는 중요 한 것이 
2. score=14.674 file=국심-2000-서-2316.pdf chunk=8
    부가가치세 과세대상이 되는지 여부에 대해서는 어떠한 안내도 받지 못하였던 바,  처분청이 쟁점전화서비

-- P2 retrieval log 일부 (파싱된 15건) --
## Retrieved Issues
1. score=10.969 file=조심-2015-부-3287.pdf issue=1
   쟁점: 청구인이 쟁점세금계산서를 수취함에 있어 선의의 거래당사자에 해당하는지 여부
   결론: 청구인은 선의의 거래당사자라고 보기 어렵고, 쟁점세금계산서의 매입세액 공제를 배제한 처분은 정당하다.
2. score=9.260 file=조심-2017-구-0609.pdf issue=1
   쟁점: 매입누락액에 부가가치율을 적용하여 매출누락액을 환산한 뒤 부가가치세를 과세한 처분이 근거과세 원칙에 위배되는지 여부
   결론: 기각 — 부가가치율을 적용한 매출누락액 환산 및 부가가치세 과세는 정당하다.
3. score=9.194 file=조심-2010-중-3848.pdf issue=0
   

-- P3 retrieval log 일부 (파싱된 3건) --
## Seed Phrase Nodes
1. score=18.71 type=LegalConcept label=실제 거래
2. score=13.72 type=Leg

**Score 분포 / 확신도**

| 파이프라인 | top-1 score | top-5 평균 | margin | 검색 건수 | latency(s) |
|---|---|---|---|---|---|
| P1 | 17.191 | 14.238 | 2.953 | 10 | 20.3 |
| P2 | 10.969 | 9.466 | 1.503 | 15 | 16.4 |
| P3 | 26.23 | 23.343 | 2.887 | 3 | 20.4 |

**사건 겹침도 (Jaccard)**

| 비교 | 겹친 사건 수 | Jaccard |
|---|---|---|
| P1 ∩ P2 | 0 | 0.00 |
| P1 ∩ P3 | 0 | 0.00 |
| P2 ∩ P3 | 0 | 0.00 |

In [16]:
metrics3 = compute_metrics(results3, show_log_excerpt=True)

-- P1 retrieval log 일부 (파싱된 10건) --
## Retrieved Chunks
1. score=12.379 file=조심-2025-구-0998.pdf chunk=3
   하여 정산하고 세금계산서를 수취하고 다 결국 당사자간 수수료를 확정함에 있어 정산서는 가장 중요한 최종 증빙으로서 해당 정산서는 세금계 .  서 수수내역와 일치하며관련 금융증빙을 통하여 실제 입출금내역을 확인할 수 있다 ,  . 나    ( ) 국내 면세점에서 상품을 구매하는 따이공은 어느 특정여행사에 계속하여 종속되는 가이드나 직 신분이 아니다 이들 따이공은 가이드와의 친분이나 구매대행업을
2. score=12.015 file=조심-2017-구-0609.pdf chunk=1
   7.1.13.  제기하였다. 청구인 주장 및 처분청 의견 2.  가 청구인 주장 .       처분청은

-- P2 retrieval log 일부 (파싱된 15건) --
## Retrieved Issues
1. score=8.727 file=조심-2017-서-0357.pdf issue=0
   쟁점: 쟁점거래처로부터 쟁점세금계산서상 공급가액 상당의 실물 거래가 있었는지 여부
   결론: 청구주장을 받아들이지 아니하고, 쟁점거래처와 쟁점세금계산서상 공급가액 상당의 실제 거래가 없다고 보아 매입세액 불공제 처분을 유지하였다.
2. score=8.129 file=조심-2019-전-3818.pdf issue=2
   쟁점: 세무조사 후 직권으로 감액경정된 세액에 대해 제기한 심판청구가 적법한지 여부
   결론: 각하. 직권 감액경정으로 불복대상이 없어 심판청구의 이 부분은 부적법하다고 판단하였다.
3. score=7.787 file=조심-2017-구-0609.pdf issue=1


-- P3 retrieval log 일부 (파싱된 5건) --
## Seed Phrase Nodes
1. score=18.59 type=EvidenceType label=세무조사
2. score=18.59 type=Lega

**Score 분포 / 확신도**

| 파이프라인 | top-1 score | top-5 평균 | margin | 검색 건수 | latency(s) |
|---|---|---|---|---|---|
| P1 | 12.379 | 10.888 | 1.491 | 10 | 27.1 |
| P2 | 8.727 | 7.468 | 1.259 | 15 | 13.7 |
| P3 | 33.14 | 29.246 | 3.894 | 5 | 22.4 |

**사건 겹침도 (Jaccard)**

| 비교 | 겹친 사건 수 | Jaccard |
|---|---|---|
| P1 ∩ P2 | 2 | 0.18 |
| P1 ∩ P3 | 2 | 0.18 |
| P2 ∩ P3 | 1 | 0.11 |

### 6-6. 빈 템플릿 — 여기에 질문만 바꿔서 계속 테스트 가능


In [ ]:
# 특정 파이프라인만 비교하고 싶을 때 (예: 2번과 3번만)
# results = compare("여기에 질문을 넣으세요", which=(2, 3))
# compute_metrics(results)

In [ ]:
# top-k 와 graph 모드를 바꿔서 실험
# results = compare("여기에 질문을 넣으세요", top_k_raw=15, top_k_issue=8, graph_mode="overview")
# compute_metrics(results)